# CS781 Phase 5 — Legal Statute Identification (LSI)

**Task:** multi-label document classification — each case may cite **several** of the 100 statute IDs.
Threshold selection, rare statutes, class balancing and long-document aggregation are central.

**Model:** chunks `[B·C, L]` (first `MAX_CHUNKS` overlapping 512-token windows of the case) → Phase-1 backbone `encode()` →
token pooling per chunk (`last_token` / `mean` / `attention`) → document pooling over real chunks (`mean` / `max` / `attention`) →
linear head → 100 logits, trained with `BCEWithLogitsLoss` (optional `pos_weight`). A statute is predicted when
`sigmoid(logit) ≥ threshold`; the threshold is tuned on dev before test inference.

**Output (`prediction.json`):** a JSON **list** with exactly one object per test case; `pred` is a list of **unique integer** statute IDs:

```json
[{"id": "case_001", "pred": [1, 7, 23]}, {"id": "case_002", "pred": [4]}]
```

**Primary metric:** Macro-F1 (secondary: Micro-F1, Subset Accuracy).

**Adaptation techniques**

| # | Key | Technique | Implementation |
|---|-----|-----------|----------------|
| 1 | `full`    | Full fine-tuning | all backbone + head parameters trainable |
| 2 | `partial` | Partial fine-tuning | backbone modules listed in `cfg.partial_layers` + head |
| 3 | `lorma`   | **LORMA (compulsory)** — low-rank **multiplicative** update `(I + s·BA)·W₀x` | Custom `LowRankMulLinear` injection |
| 4 | `kd`      | **Knowledge Distillation (compulsory)** | `DistillationExperiment` + `multilabel_kd_loss` (section 15) |

**How to run.** Set paths and run flags in section 2, then *Run All*. The default flags reproduce the best submission:
preflight checks → train `exp_cfg` (CNN + LORMA, mean token pooling) → dev threshold tuning → test inference → `submission.zip`.
Knowledge distillation runs only when `RUN_KD = True`.

| Stage | Section |
|---|---|
| Environment and imports | 1 |
| Paths and configuration | 2 |
| Data loading and validation | 3 |
| Tokenizer | 4 |
| Dataset and DataLoaders | 5 |
| Phase-1 backbone definitions | 6 |
| Backbone registry and checkpoint checks | 7 |
| Phase-5 model wrapper | 8 |
| Pooling and class balancing | 9 |
| Adaptation methods | 10 |
| Experiment configuration | 11 |
| Training, evaluation and submission utilities | 12 |
| Preflight tests (optional, lightweight) | 13 |
| Main training experiment | 14 |
| Knowledge distillation (optional) | 15 |
| Test inference (dev-tuned threshold) | 16 |
| Submission validation and ZIP | 17 |

## 1. Environment and imports

In [ ]:
!pip install -q huggingface_hub safetensors peft

In [ ]:
import os, json, random, math, time, zipfile, copy, gc
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, field, replace
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

# Built-in PEFT support (LoRA, rsLoRA, IA3)
import peft
from peft import LoraConfig, IA3Config, get_peft_model

print("torch:", torch.__version__)
print("peft :", peft.__version__)

# ------------------------------------------------------------
# Determinism
# ------------------------------------------------------------
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 2. Paths and configuration

All Kaggle paths and run flags live here. Nothing below hard-codes a checkpoint path.

In [ ]:
# ------------------------------------------------------------
# Phase-1 tokenizer and backbone checkpoints
# ------------------------------------------------------------
TOKENIZER_PATH     = "/kaggle/input/datasets/partnerincrime/tookkenizer/legal_bpe.json"
# Phase-1 decoder backbone (plain state dict; a trained Phase-5 checkpoint does not load into the bare backbone).
DECODER_CHECKPOINT = "/kaggle/input/datasets/sg123321/decscratch/pytorch_model.bin"
SSM_CHECKPOINT     = "/kaggle/input/datasets/sg123321/ssmonly/model.safetensors"
ENCDEC_CHECKPOINT  = "/kaggle/input/datasets/sg123321/converted-encdec-scratch"   # directory or file
CNN_CHECKPOINT     = "/kaggle/input/datasets/cs781ner/cnnnattn/model.safetensors"

# Knowledge distillation (section 15)
KD_TEACHER_CHECKPOINT = "/kaggle/input/datasets/partnerincrime/secfull55/dec full.pt"   # trained decoder + full FT
# Student initialisation: first STUDENT_LAYERS blocks of this checkpoint (a Phase-1 backbone or a Phase-5
# classification checkpoint with "backbone." keys).
KD_STUDENT_INIT_SOURCE = DECODER_CHECKPOINT
# Optional: continue from an existing KD student checkpoint instead. None = start from KD_STUDENT_INIT_SOURCE.
KD_STUDENT_RESUME_CHECKPOINT = None

# Checkpoint used by test inference when the model was not trained in this session.
# None -> <run_dir>/best_model.pt of the selected configuration.
INFERENCE_CHECKPOINT = None

# ------------------------------------------------------------
# Long-document chunking / loader settings
# ------------------------------------------------------------
CHUNK_TOKENS = 512       # tokens per chunk incl. BOS/EOS (<= backbone context length)
CHUNK_STRIDE = 384       # < CHUNK_TOKENS - 2  => overlapping windows
MAX_CHUNKS = 16          # first MAX_CHUNKS chunks kept per case
BATCH_SIZE = 2           # cases per batch
NUM_WORKERS = 0          # 0 is the safe default on Kaggle

# ------------------------------------------------------------
# What a "Run All" executes (defaults reproduce the best submission: CNN + LORMA)
# ------------------------------------------------------------
RUN_PREFLIGHT = True                                      # section 13 (lightweight)
PREFLIGHT_BACKBONES = ["decoder", "encdec", "cnn", "ssm"]  # backbones checked in section 13
RUN_TINY_OVERFIT = False                                  # section 13
RUN_TRAINING = True                                       # section 14: trains exp_cfg
RUN_KD = False                                            # section 15
RUN_INFERENCE = True                                      # sections 16-17
INFERENCE_SOURCE = "main"                                 # "main" = exp_cfg model, "kd" = KD student

In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download, list_repo_files

HF_TOKEN = UserSecretsClient().get_secret("hf_token_rp")
login(token=HF_TOKEN, add_to_git_credential=False)

# ============================================================
# PHASE 5 DATASET FROM HUGGING FACE
# ============================================================
DATA_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_5_LSI"

REPO_FILES = list_repo_files(DATA_REPO, repo_type="dataset", token=HF_TOKEN)
PHASE_FILES = [f for f in REPO_FILES if f.startswith(PHASE_DIR + "/")]
print(f"{PHASE_DIR} files in repo:")
for f in PHASE_FILES:
    print("  ", f)


def get_data_file(filename):
    """Download `filename`; if that exact path is absent, fall back to a UNIQUE basename match inside PHASE_DIR."""
    if filename not in REPO_FILES:
        base = filename.split("/")[-1]
        matches = [f for f in PHASE_FILES if f.split("/")[-1] == base]
        if len(matches) != 1:
            raise FileNotFoundError(f"'{filename}' not in repo; basename matches in {PHASE_DIR}: {matches}")
        print(f"[note] '{filename}' not found, using '{matches[0]}'")
        filename = matches[0]
    return Path(
        hf_hub_download(
            repo_id=DATA_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    )


TRAIN_FILE = get_data_file(
    "Phase_5_LSI/Train/train.json"
)

DEV_FILE = get_data_file(
    "Phase_5_LSI/Train/dev.json"
)

TEST_FILE = get_data_file(
    "Phase_5_LSI/Test/test.json"
)

for name, p in [
    ("TRAIN_FILE", TRAIN_FILE),
    ("DEV_FILE", DEV_FILE),
    ("TEST_FILE", TEST_FILE),
]:
    print(f"{name:16s}: {p}")

## 3. Data loading and validation

Inspect the raw JSON first. Each train/dev case has an id, its text and a `"labels"` list of statute IDs; test has id + text only.
The label space (`NUM_CLASSES`, `statute2idx`, `id2statute`) is built from the **training set**.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def describe_json(obj, name, depth=0, max_items=2):
    """Print the nested structure of a JSON object (types, sizes, first keys) to discover the schema."""
    pad = "  " * depth
    if isinstance(obj, dict):
        print(f"{pad}{name}: dict[{len(obj)}] keys={list(obj.keys())[:8]}")
        for k in list(obj.keys())[:max_items]:
            describe_json(obj[k], repr(k), depth + 1, max_items)
    elif isinstance(obj, list):
        print(f"{pad}{name}: list[{len(obj)}]")
        for i, x in enumerate(obj[:max_items]):
            describe_json(x, f"[{i}]", depth + 1, max_items)
    else:
        s = repr(obj)
        print(f"{pad}{name}: {s[:120]}{'...' if len(s) > 120 else ''}")


train_raw = load_json(TRAIN_FILE)
dev_raw   = load_json(DEV_FILE)
test_raw  = load_json(TEST_FILE)

describe_json(train_raw, "train")
describe_json(dev_raw, "dev")
describe_json(test_raw, "test")

In [ ]:
train_ids = {x for row in train_raw for x in row["labels"]}
dev_ids = {x for row in dev_raw for x in row["labels"]}

unseen = dev_ids - train_ids

print("Train statutes:", len(train_ids))
print("Dev statutes:", len(dev_ids))
print("Unseen dev statutes:", len(unseen))
print("Unseen IDs:", sorted(unseen))

In [ ]:
def normalize_split(raw, has_labels=True):
    """
    Normalize raw JSON records into a uniform list of cases.

    Each labeled case:
        {"id": str, "text": str, "labels": List[int]}

    Each test case:
        {"id": str, "text": str}
    """
    normalized = []

    for record in raw:
        # Preserve ID exactly as given
        case_id = record["id"]

        # Preserve document text
        text = record["text"]

        case = {
            "id": case_id,
            "text": text
        }

        if has_labels:
            # Convert statute IDs to integers, deduplicate, and sort
            labels = sorted(set(int(label) for label in record["labels"]))
            case["labels"] = labels

        normalized.append(case)

    return normalized


def build_statute_vocab(train_cases):
    """
    Build the global statute vocabulary using training labels only.

    Returns:
        STATUTE_IDS: sorted list of unique statute IDs
        statute2idx: statute ID -> class index
        id2statute: class index -> statute ID
    """
    # Collect unique statute IDs from training cases
    statute_ids = set()

    for case in train_cases:
        statute_ids.update(case["labels"])

    # Sorted list of statute IDs
    STATUTE_IDS = sorted(statute_ids)

    # Map statute IDs to consecutive class indices
    statute2idx = {
        statute_id: idx
        for idx, statute_id in enumerate(STATUTE_IDS)
    }

    # Reverse mapping
    id2statute = {
        idx: statute_id
        for idx, statute_id in enumerate(STATUTE_IDS)
    }

    return STATUTE_IDS, statute2idx, id2statute


# Normalize all splits
train_cases = normalize_split(train_raw)
dev_cases   = normalize_split(dev_raw)
test_cases  = normalize_split(test_raw, has_labels=False)

# Build vocabulary from training data ONLY
STATUTE_IDS, statute2idx, id2statute = build_statute_vocab(train_cases)

NUM_CLASSES = len(STATUTE_IDS)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

assert all(type(s) is int for s in STATUTE_IDS), \
    "Statute IDs must be Python integers"

assert len(statute2idx) == len(id2statute) == NUM_CLASSES

assert all("labels" not in case for case in test_cases), \
    "Test cases must not contain labels"

assert all(
    len(case["labels"]) == len(set(case["labels"]))
    for case in train_cases + dev_cases
), "Duplicate labels detected"

# Report development labels absent from training
train_statute_set = set(STATUTE_IDS)

dev_statute_set = {
    label
    for case in dev_cases
    for label in case["labels"]
}

unseen_dev_labels = sorted(dev_statute_set - train_statute_set)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("Train cases:", len(train_cases))
print("Dev cases  :", len(dev_cases))
print("Test cases :", len(test_cases))
print("NUM_CLASSES:", NUM_CLASSES)
print("Unseen dev labels:", unseen_dev_labels)
print("Number of unseen dev labels:", len(unseen_dev_labels))

In [ ]:
# ------------------------------------------------------------
# Data validation (raises on any inconsistency)
# ------------------------------------------------------------
def validate_cases(cases, raw, name, has_labels):
    ids = [c["id"] for c in cases]
    assert len(ids) == len(set(ids)), f"{name}: duplicate case IDs"
    assert ids == [r["id"] for r in raw], f"{name}: case IDs lost, added or reordered"
    for c in cases:
        assert isinstance(c["text"], str), c["id"]
        if has_labels:
            assert c["labels"], f"{name}/{c['id']}: case without statutes"
            unknown = set(c["labels"]) - set(STATUTE_IDS)
            assert not unknown, f"{name}/{c['id']}: statutes outside the training vocabulary {sorted(unknown)}"
        else:
            assert "labels" not in c, f"{name}/{c['id']}: test case carries labels"
    print(f"{name:5s}: {len(cases)} cases - OK")


validate_cases(train_cases, train_raw, "train", has_labels=True)
validate_cases(dev_cases, dev_raw, "dev", has_labels=True)
validate_cases(test_cases, test_raw, "test", has_labels=False)
assert STATUTE_IDS == sorted(STATUTE_IDS) and len(STATUTE_IDS) == NUM_CLASSES

In [ ]:
def label_frequency(cases):
    cnt = Counter()
    for c in cases:
        cnt.update(c["labels"])
    return cnt


train_freq = label_frequency(train_cases)
dev_freq = label_frequency(dev_cases)

labels_per_case = [len(c["labels"]) for c in train_cases]
print("labels/case: mean %.2f | max %d | cases with 0 labels: %d"
      % (np.mean(labels_per_case), max(labels_per_case), sum(1 for n in labels_per_case if n == 0)))
print("Most frequent statutes:", train_freq.most_common(10))
print("Statutes with < 5 train cases:", sum(1 for s in STATUTE_IDS if train_freq[s] < 5))

unseen_dev = set(dev_freq) - set(STATUTE_IDS)
print("Dev statutes unseen in train:", len(unseen_dev), sorted(unseen_dev)[:20])
assert not ({c["id"] for c in train_cases} & {c["id"] for c in dev_cases}), "train/dev id overlap"

## 4. Tokenizer

Same `ByteLevelBPE` class and `legal_bpe.json` as Phase 3 (unchanged).

In [ ]:
#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer


import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

In [ ]:
tokenizer = ByteLevelBPE.load(TOKENIZER_PATH)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

print("PAD/UNK/BOS/EOS:", PAD_ID, UNK_ID, BOS_ID, EOS_ID)

_probe = "The appellant filed a petition under Section 9 of the Act."
_ids = tokenizer.encode(_probe)
print(len(_ids), _ids[:20])
print(repr(tokenizer.decode(_ids)))

## 5. Dataset and DataLoaders

**Batch contract** expected by the training loops (sections 12 & 15) and the model (section 8):

| key | shape | notes |
|-----|-------|-------|
| `input_ids` | `[B, C, L]` long | token ids per chunk, `PAD_ID` padded |
| `attention_mask` | `[B, C, L]` long | 1 = real token |
| `chunk_mask` | `[B, C]` bool | 1 = real chunk |
| `labels` | `[B, NUM_CLASSES]` float | multi-hot targets (all zeros for test) |
| `has_labels` | `[B]` bool | False for test items |
| `ids` | `List[str]` | case ids, not moved to device |

`chunk_token_ids()` is a sliding-window helper; `LSI_Dataset` keeps the **first** `MAX_CHUNKS` chunks of each case.

In [ ]:
def chunk_token_ids(ids, chunk_len, stride, max_chunks=None, add_bos_eos=True):
    """
    Generic sliding-window chunker (no task logic).
      ids        : token ids of one full document (no BOS/EOS)
      chunk_len  : tokens per chunk INCLUDING BOS/EOS when add_bos_eos=True
                   (must be <= BACKBONE_MAX_TOKENS[backbone] — 1024 for encdec)
      stride     : step between chunk starts (stride < body length => overlap)
      max_chunks : keep at most this many chunks (None = all). Selection policy
                   (first-N / evenly spaced / head+tail) is a Phase-specific choice.
    Returns List[List[int]] (always >= 1 chunk, even for an empty document).
    """
    body = chunk_len - 2 if add_bos_eos else chunk_len
    assert body > 0 and 0 < stride <= body, (chunk_len, stride)
    chunks = []
    start = 0
    while True:
        piece = ids[start:start + body]
        chunks.append([BOS_ID] + piece + [EOS_ID] if add_bos_eos else piece)
        if start + body >= len(ids):
            break
        start += stride
    if max_chunks is not None:
        chunks = chunks[:max_chunks]
    return chunks


print(chunk_token_ids(list(range(10, 25)), chunk_len=8, stride=4))

In [ ]:
def to_multi_hot(statute_ids, statute2idx, num_classes):
    """List of statute IDs -> float multi-hot vector [num_classes]."""
    target = torch.zeros(num_classes, dtype=torch.float32)

    for statute_id in statute_ids:
        statute_id = int(statute_id)
        if statute_id in statute2idx:
            target[statute2idx[statute_id]] = 1.0

    return target


class LSI_Dataset(Dataset):
    """
    One item = one case.

    Returns:
        {
            "id": original case ID,
            "chunks": List[List[int]],
            "target": FloatTensor [NUM_CLASSES],
            "has_labels": bool,
        }
    """

    def __init__(
        self,
        cases,
        tokenizer,
        statute2idx,
        chunk_tokens=CHUNK_TOKENS,
        chunk_stride=CHUNK_STRIDE,
        max_chunks=MAX_CHUNKS,
        is_test=False,
    ):
        self.items = []

        for case in cases:
            case_id = case["id"]
            text = case["text"]

            # Tokenize each document exactly once.
            # Adjust this line if your tokenizer uses a different API.
            token_ids = tokenizer.encode(text)

            if isinstance(token_ids, torch.Tensor):
                token_ids = token_ids.tolist()

            # Use the existing chunking implementation.
            chunks = chunk_token_ids(
                token_ids,
                chunk_len=chunk_tokens,
                stride=chunk_stride,
            )

            # Keep the first max_chunks chunks.
            chunks = [list(chunk) for chunk in chunks[:max_chunks]]

            # Every document must have at least one chunk.
            if not chunks:
                chunks = [[BOS_ID]]

            # Test records have no labels.
            has_labels = not is_test and "labels" in case

            if has_labels:
                target = to_multi_hot(
                    case["labels"],
                    statute2idx,
                    NUM_CLASSES,
                )
            else:
                target = torch.zeros(NUM_CLASSES, dtype=torch.float32)

            self.items.append({
                "id": case_id,
                "chunks": chunks,
                "target": target,
                "has_labels": has_labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def lsi_collator(batch):
    """
    Dynamically pad chunks and token sequences.
    """
    batch_size = len(batch)

    max_chunk_count = max(
        1, max(len(item["chunks"]) for item in batch)
    )

    max_token_length = max(
        1,
        max(
            len(chunk)
            for item in batch
            for chunk in item["chunks"]
        ),
    )

    input_ids = torch.full(
        (batch_size, max_chunk_count, max_token_length),
        PAD_ID,
        dtype=torch.long,
    )

    attention_mask = torch.zeros(
        (batch_size, max_chunk_count, max_token_length),
        dtype=torch.long,
    )

    chunk_mask = torch.zeros(
        (batch_size, max_chunk_count),
        dtype=torch.bool,
    )

    labels = torch.stack([
        item["target"].to(torch.float32)
        for item in batch
    ])

    has_labels = torch.tensor(
        [item["has_labels"] for item in batch],
        dtype=torch.bool,
    )

    ids = [item["id"] for item in batch]

    for i, item in enumerate(batch):
        chunks = item["chunks"]

        for j, chunk in enumerate(chunks):
            chunk_length = len(chunk)

            if chunk_length == 0:
                continue

            input_ids[i, j, :chunk_length] = torch.tensor(
                chunk, dtype=torch.long
            )

            attention_mask[i, j, :chunk_length] = 1
            chunk_mask[i, j] = 1

        # Padded chunk slots: BOS is attended to, but the chunk
        # itself remains masked out.
        for j in range(len(chunks), max_chunk_count):
            input_ids[i, j, 0] = BOS_ID
            attention_mask[i, j, 0] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "chunk_mask": chunk_mask,
        "labels": labels,
        "has_labels": has_labels,
        "ids": ids,
    }

In [ ]:
# ============================================================
# DataLoaders (boilerplate)
# ============================================================

def build_dataloaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    # num_workers=0: items are pre-tokenized in __init__; avoids the worker-shutdown warnings seen in Phase 3
    kw = dict(chunk_tokens=CHUNK_TOKENS, chunk_stride=CHUNK_STRIDE, max_chunks=MAX_CHUNKS)
    train_ds = LSI_Dataset(train_cases, tokenizer, statute2idx, **kw)
    dev_ds   = LSI_Dataset(dev_cases, tokenizer, statute2idx, **kw)
    test_ds  = LSI_Dataset(test_cases, tokenizer, statute2idx, is_test=True, **kw)

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=lsi_collator, num_workers=num_workers, generator=g)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size, shuffle=False,
                            collate_fn=lsi_collator, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=lsi_collator, num_workers=num_workers)
    return train_ds, dev_ds, test_ds, train_loader, dev_loader, test_loader


train_ds, dev_ds, test_ds, train_loader, dev_loader, test_loader = build_dataloaders()

print("Train items:", len(train_ds))
print("Dev items  :", len(dev_ds))
print("Test items :", len(test_ds))
assert len(test_ds) == len(test_cases), "every test case must be an item"

_b = next(iter(dev_loader))
for k, v in _b.items():
    print(f"{k:16s}", tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)

## 6. Phase-1 backbone definitions

Every backbone exposes `encode(input_ids, attention_mask=None) -> [B, L, D]`; the Phase-5 wrapper calls it on chunks `[B·C, L]`.
Checkpoint paths are set in section 2.

| key | architecture | hidden dim | max tokens | representation used |
|-----|--------------|-----------:|-----------:|---------------------|
| `decoder` | decoder-only causal Transformer | 640 | 2048 | causal token states after `final_ln` |
| `ssm` | Diagonal SSM / S4D-inspired LM | 700 | 2048 | causal token states after `norm_f` |
| `encdec` | encoder-decoder Transformer | 512 | 1024 (input) | bidirectional **encoder** states |
| `cnn` | CNN + causal self-attention hybrid | 640 | 2048 | causal token states after `final_norm` |

### 6A. Decoder-only Transformer

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderOnlyMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderOnlyBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderOnlyMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderOnlyBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }

In [ ]:
decoder_config = DecoderConfig(
    vocab_size=16000,
    context_length=2048,
    n_embd=640,
    n_layer=8,
    n_head=8,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)

### 6B. Diagonal SSM / S4D-inspired LM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SSMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim,
        n_layer,
        max_seq_len,
        state_dim=16,
        expand=3,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.layers = nn.ModuleList([
            SSMBlock(
                dim=dim,
                state_dim=state_dim,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.norm_f = nn.LayerNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        labels=None,
        **kwargs
    ):
        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.max_seq_len}"
            )

        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        logits = self.lm_head(x)

        if labels is not None:

            targets = torch.full_like(
                labels,
                -100
            )

            targets[:, :-1] = labels[:, 1:]

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    self.vocab_size
                ),
                targets.reshape(-1),
                ignore_index=-100,
            )

            return {
                "loss": loss
            }

        return {
            "logits": logits
        }
class SSMBlock(nn.Module):
    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.ssm = DiagonalSSM(
            dim=dim,
            state_dim=state_dim,
            expand=expand,
        )

    def forward(self, x):
        return x + self.ssm(self.norm(x))
class DiagonalSSM(nn.Module):
    """
    S4D-inspired diagonal state-space layer.

    Input : [B, T, C]
    Output: [B, T, C]
    """

    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.dim = dim
        self.state_dim = state_dim
        self.inner_dim = dim * expand

        # content + gate
        self.in_proj = nn.Linear(
            dim,
            2 * self.inner_dim
        )

        # Stable diagonal state transition
        # A will become negative
        A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32
        )

        self.A_log = nn.Parameter(
            torch.log(A)
        )

        # Per-channel SSM input/output weights
        self.B = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        self.C = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        # Per-channel time scale
        self.log_dt = nn.Parameter(
            torch.zeros(self.inner_dim)
        )

        # Direct skip term
        self.D = nn.Parameter(
            torch.ones(self.inner_dim)
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            dim
        )

    def forward(self, x):
        Bsz, T, _ = x.shape

        xz = self.in_proj(x)

        u, z = xz.chunk(
            2,
            dim=-1
        )

        # -------------------------------------------------
        # Step 1
        #
        # Construct stable continuous transition:
        #
        # A = -exp(A_log)
        #
        # shape:
        # [state_dim]
        # -------------------------------------------------

        A = -torch.exp(self.A_log)

        # -------------------------------------------------
        # Step 2
        #
        # dt must always be positive.
        #
        # use softplus(log_dt)
        #
        # shape:
        # [inner_dim]
        # -------------------------------------------------

        dt = F.softplus(self.log_dt)

        # -------------------------------------------------
        # Step 3
        #
        # Create discrete transition:
        #
        # A_bar[c,n] = exp(dt[c] * A[n])
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        A_bar = torch.exp(
            dt.unsqueeze(-1) * A.unsqueeze(0)
        )

        # -------------------------------------------------
        # Step 4
        #
        # Zero-order-hold style input coefficient:
        #
        # B_bar[c,n]
        #   = ((A_bar[c,n] - 1) / A[n]) * B[c,n]
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        xA = dt.unsqueeze(-1) * A.unsqueeze(0)

        B_bar = (
            torch.expm1(xA) / A.unsqueeze(0)
        ) * self.B

        # -------------------------------------------------
        # Step 5
        #
        # Build time indices:
        #
        # t = [0, 1, ..., T-1]
        #
        # Then construct the impulse-response kernel:
        #
        # K[c,t] =
        #   sum_n C[c,n]
        #         * A_bar[c,n]^t
        #         * B_bar[c,n]
        #
        # Target shape:
        # [inner_dim, T]
        #
        # Hint:
        # avoid a Python loop over T.
        # Broadcasting / einsum should be used.
        # -------------------------------------------------

        t = torch.arange(
            T,
            device=x.device,
            dtype=A_bar.dtype
        )

        powers = A_bar.unsqueeze(-1) ** t
        # [inner_dim, state_dim, T]

        kernel = torch.einsum(
            "cn,cnt->ct",
            self.C * B_bar,
            powers
        )
        # [inner_dim, T]

        # -------------------------------------------------
        # Step 6 — FFT causal convolution
        #
        # u currently:
        # [B, T, inner_dim]
        #
        # transpose to:
        # [B, inner_dim, T]
        #
        # Use an FFT length >= 2*T.
        #
        # rfft(input)
        # rfft(kernel)
        # multiply
        # irfft
        #
        # finally truncate to first T values.
        #
        # This gives causal convolution:
        #
        # y[t] = sum_{i=0}^{t} K[i] u[t-i]
        # -------------------------------------------------

        u = u.transpose(1, 2)
        # [B, inner_dim, T]

        fft_len = 2 * T

        u_fft = u.float()
        kernel_fft = kernel.float()
        
        u_f = torch.fft.rfft(
            u_fft,
            n=fft_len
        )
        
        k_f = torch.fft.rfft(
            kernel_fft,
            n=fft_len
        )
        
        y = torch.fft.irfft(
            u_f * k_f.unsqueeze(0),
            n=fft_len
        )[..., :T]
        
        y = y.to(u.dtype)

        # -------------------------------------------------
        # Step 7
        #
        # Add direct SSM skip:
        #
        # y += D * u
        # -------------------------------------------------

        y = y + self.D.view(1, -1, 1) * u

        # restore:
        # [B, T, inner_dim]

        y = y.transpose(1, 2)

        # same gating idea as before
        y = y * F.silu(z)

        return self.out_proj(y)

In [ ]:
def ssm_encode(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > self.max_seq_len:
        raise ValueError(
            f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}"
        )

    x = self.embedding(input_ids)

    for layer in self.layers:
        x = layer(x)

    x = self.norm_f(x)

    return x

SSMLM.encode = ssm_encode

In [ ]:
SSM_DIM = 700
SSM_N_LAYER = 10
SSM_MAX_SEQ_LEN = 2048
SSM_STATE_DIM = 16
SSM_EXPAND = 3


def build_ssm_backbone():
    return SSMLM(
        vocab_size=16000,
        dim=SSM_DIM,
        n_layer=SSM_N_LAYER,
        max_seq_len=SSM_MAX_SEQ_LEN,
        state_dim=SSM_STATE_DIM,
        expand=SSM_EXPAND,
    )

### 6C. Encoder-decoder Transformer

Classification uses the bidirectional **encoder** outputs (`encode()`); inputs must be ≤ `max_input_length` = 1024 tokens.

In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class EncoderDecoderConfig:
    vocab_size: int
    d_model: int
    d_kv: int
    d_ff: int
    num_layers: int
    num_decoder_layers: int
    num_heads: int
    dropout_rate: float = 0.1
    layer_norm_epsilon: float = 1e-6
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    feed_forward_proj: str = "relu"
    dense_act_fn: str = "relu"
    tie_word_embeddings: bool = True
    scale_decoder_outputs: bool = True
    initializer_factor: float = 1.0
    pad_token_id: int = 0
    eos_token_id: int = 3
    decoder_start_token_id: int = 0
    max_input_length: int = 1024
    max_target_length: int = 1024
    use_cache: bool = False

    @property
    def inner_dim(self):
        return self.num_heads * self.d_kv

    @property
    def is_gated_act(self):
        return self.feed_forward_proj.startswith("gated-")


class RMSNorm(nn.Module):
    """Scale-only RMS normalization (no mean subtraction, no bias)."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.float().pow(2).mean(dim=-1, keepdim=True)
        normalized = hidden_states * torch.rsqrt(
            variance + self.variance_epsilon
        ).to(dtype=hidden_states.dtype)
        return normalized * self.weight.to(dtype=hidden_states.dtype)


class RelativePositionBias(nn.Module):
    """Bucketed relative-position bias: log-spaced buckets, one learned bias per head."""

    def __init__(
        self,
        num_buckets,
        num_heads,
        max_distance,
        bidirectional,
    ):
        super().__init__()
        if num_buckets < 2:
            raise ValueError("relative_attention_num_buckets must be >= 2")
        if max_distance < 2:
            raise ValueError("relative_attention_max_distance must be >= 2")
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def relative_position_bucket(
        relative_position,
        bidirectional=True,
        num_buckets=32,
        max_distance=128,
    ):
        """Map key_position - query_position to a relative-position bucket."""
        bucket = 0
        if bidirectional:
            num_buckets //= 2
            bucket = bucket + (
                (relative_position > 0).to(torch.long) * num_buckets
            )
            relative_position = torch.abs(relative_position)
        else:
            relative_position = -torch.minimum(
                relative_position, torch.zeros_like(relative_position)
            )

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        relative_position_if_large = max_exact + (
            torch.log(
                relative_position.float().clamp(min=1) / max_exact
            )
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        relative_position_if_large = torch.clamp(
            relative_position_if_large, max=num_buckets - 1
        )
        bucket = bucket + torch.where(
            is_small, relative_position, relative_position_if_large
        )
        return bucket

    def forward(self, query_length, key_length, device=None):
        if device is None:
            device = self.embedding.weight.device
        query_positions = torch.arange(
            query_length, dtype=torch.long, device=device
        )[:, None]
        key_positions = torch.arange(
            key_length, dtype=torch.long, device=device
        )[None, :]
        relative_position = key_positions - query_positions
        buckets = self.relative_position_bucket(
            relative_position,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance,
        )
        values = self.embedding(buckets)
        return values.permute(2, 0, 1).unsqueeze(0)


class MultiHeadAttention(nn.Module):
    """Multi-head attention with bias-free Q/K/V/O projections."""

    def __init__(
        self,
        config: EncoderDecoderConfig,
        causal=False,
        relative_bias_bidirectional=None,
    ):
        super().__init__()
        self.d_model = config.d_model
        self.d_kv = config.d_kv
        self.num_heads = config.num_heads
        self.inner_dim = config.inner_dim
        self.causal = causal

        self.q = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relative_attention_bias = None
        if relative_bias_bidirectional is not None:
            self.relative_attention_bias = RelativePositionBias(
                num_buckets=config.relative_attention_num_buckets,
                num_heads=config.num_heads,
                max_distance=config.relative_attention_max_distance,
                bidirectional=relative_bias_bidirectional,
            )

    def _split_heads(self, projected):
        batch_size, seq_len, _ = projected.shape
        return projected.view(
            batch_size, seq_len, self.num_heads, self.d_kv
        ).transpose(1, 2)

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        attention_mask=None,
        position_bias=None,
    ):
        if key_value_states is None:
            key_value_states = hidden_states
        query = self._split_heads(self.q(hidden_states))
        key = self._split_heads(self.k(key_value_states))
        value = self._split_heads(self.v(key_value_states))

        scores = torch.matmul(query, key.transpose(-2, -1))
        query_length, key_length = scores.shape[-2:]

        if position_bias is None and self.relative_attention_bias is not None:
            position_bias = self.relative_attention_bias(
                query_length, key_length, device=scores.device
            )
        if position_bias is not None:
            scores = scores + position_bias.to(dtype=scores.dtype)

        if self.causal:
            if query_length != key_length:
                raise ValueError(
                    "This implementation does not use a KV cache; "
                    "causal self-attention requires equal query/key lengths."
                )
            causal_allowed = torch.ones(
                query_length,
                key_length,
                dtype=torch.bool,
                device=scores.device,
            ).tril()
            scores = scores.masked_fill(
                ~causal_allowed.view(1, 1, query_length, key_length),
                torch.finfo(scores.dtype).min,
            )

        if attention_mask is not None:
            if attention_mask.ndim != 2 or attention_mask.shape[-1] != key_length:
                raise ValueError(
                    "attention_mask must have shape [batch, key_length]"
                )
            key_is_valid = attention_mask.to(
                device=scores.device, dtype=torch.bool
            ).view(scores.shape[0], 1, 1, key_length)
            scores = scores.masked_fill(
                ~key_is_valid, torch.finfo(scores.dtype).min
            )

        probabilities = F.softmax(scores.float(), dim=-1).to(
            dtype=scores.dtype
        )
        probabilities = self.dropout(probabilities)
        attended = torch.matmul(probabilities, value)
        attended = attended.transpose(1, 2).contiguous().view(
            attended.shape[0], query_length, self.inner_dim
        )
        return self.o(attended)


class EncoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=False,
            relative_bias_bidirectional=(
                True if has_relative_attention_bias else None
            ),
        )


class DecoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=True,
            relative_bias_bidirectional=(
                False if has_relative_attention_bias else None
            ),
        )


class CrossAttention(MultiHeadAttention):
    def __init__(self, config):
        super().__init__(config, causal=False)


class EncDecFeedForward(nn.Module):
    """Feed-forward layer: dense-act-dense, or gated (two input projections)."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.feed_forward_proj = config.feed_forward_proj
        self.activation_name = config.dense_act_fn
        if config.is_gated_act:
            self.wi_0 = nn.Linear(config.d_model, config.d_ff, bias=False)
            self.wi_1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        else:
            self.wi = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.wo = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _activation(self, x):
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("gelu", "gelu_new"):
            return F.gelu(x, approximate="tanh" if self.activation_name == "gelu_new" else "none")
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported dense_act_fn: {self.activation_name}"
        )

    def forward(self, hidden_states):
        if self.feed_forward_proj.startswith("gated-"):
            hidden_states = self._activation(self.wi_0(hidden_states)) * self.wi_1(
                hidden_states
            )
        else:
            hidden_states = self._activation(self.wi(hidden_states))
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)


class EncoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = EncoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self, hidden_states, attention_mask=None, position_bias=None,
        return_debug=False
    ):
        debug = {}
        attention_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(attention_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class DecoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = DecoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.cross_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.cross_attention = CrossAttention(config)
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = EncDecFeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        position_bias=None,
        return_debug=False,
    ):
        debug = {}
        self_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=decoder_attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(self_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        cross_output = self.cross_attention(
            self.cross_norm(hidden_states),
            key_value_states=encoder_hidden_states,
            attention_mask=encoder_attention_mask,
        )
        hidden_states = hidden_states + self.dropout(cross_output)
        if return_debug:
            debug["cross_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class Encoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            EncoderBlock(config, layer_idx)
            for layer_idx in range(config.num_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(self, inputs_embeds, attention_mask=None, debug=False):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                states[f"block_{layer_idx}_self_attention"] = (
                    block_debug["self_attention_output"]
                )
                states[f"block_{layer_idx}_feed_forward"] = (
                    block_debug["feed_forward_output"]
                )
            else:
                hidden_states = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=False)

    def debug_hidden_states(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=True)


class Decoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.num_decoder_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        debug=False,
    ):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                for name, value in block_debug.items():
                    states[f"block_{layer_idx}_{name}"] = value
            else:
                hidden_states = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=False,
        )

    def debug_hidden_states(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=True,
        )


class EncoderDecoderTransformer(nn.Module):
    """Encoder-decoder Transformer, implemented with raw PyTorch layers."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        if config.d_model <= 0 or config.d_kv <= 0:
            raise ValueError("d_model and d_kv must be positive")
        if config.num_heads <= 0 or config.num_layers <= 0:
            raise ValueError("num_heads and num_layers must be positive")
        if config.num_decoder_layers <= 0:
            raise ValueError("num_decoder_layers must be positive")
        self.config = config
        self.shared_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._initialize_weights()
        if config.tie_word_embeddings:
            # One embedding matrix is shared by the encoder, decoder and LM head.
            self.lm_head.weight = self.shared_embedding.weight

    def _initialize_weights(self):
        factor = self.config.initializer_factor
        with torch.no_grad():
            self.shared_embedding.weight.normal_(mean=0.0, std=factor)
            for module in self.modules():
                if isinstance(module, RMSNorm):
                    module.weight.fill_(1.0)
                elif isinstance(module, RelativePositionBias):
                    module.embedding.weight.normal_(mean=0.0, std=factor)
                elif isinstance(module, MultiHeadAttention):
                    module.q.weight.normal_(
                        mean=0.0,
                        std=factor
                        * (self.config.d_model * self.config.d_kv) ** -0.5,
                    )
                    for projection in (module.k, module.v):
                        projection.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.o.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.inner_dim ** -0.5,
                    )
                elif isinstance(module, EncDecFeedForward):
                    if self.config.is_gated_act:
                        module.wi_0.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                        module.wi_1.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    else:
                        module.wi.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.wo.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.d_ff ** -0.5,
                    )

    def _validate_input_ids(self, input_ids, max_length, name):
        if input_ids.ndim != 2:
            raise ValueError(f"{name} must have shape [batch, sequence]")
        if input_ids.shape[1] == 0:
            raise ValueError(f"{name} must not be empty")
        if input_ids.shape[1] > max_length:
            raise ValueError(
                f"{name} length {input_ids.shape[1]} exceeds {max_length}"
            )

    def encode(self, input_ids, attention_mask=None):
        """Return final encoder states with shape [B, L, d_model]."""
        self._validate_input_ids(
            input_ids, self.config.max_input_length, "input_ids"
        )
        if attention_mask is not None and attention_mask.shape != input_ids.shape:
            raise ValueError("attention_mask must match input_ids shape")
        return self.encoder(
            self.shared_embedding(input_ids),
            attention_mask=attention_mask,
        )

    def decode(
        self,
        decoder_input_ids,
        encoder_hidden_states,
        attention_mask=None,
        decoder_attention_mask=None,
    ):
        self._validate_input_ids(
            decoder_input_ids,
            self.config.max_target_length,
            "decoder_input_ids",
        )
        if decoder_input_ids.shape[0] != encoder_hidden_states.shape[0]:
            raise ValueError(
                "decoder and encoder batch dimensions must match"
            )
        return self.decoder(
            self.shared_embedding(decoder_input_ids),
            encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )

    def shift_right(self, labels):
        if labels.ndim != 2:
            raise ValueError("labels must have shape [batch, target_length]")
        if labels.shape[1] == 0:
            raise ValueError("labels must not be empty")
        shifted = labels.new_full(labels.shape, self.config.pad_token_id)
        shifted[:, 0] = self.config.decoder_start_token_id
        shifted[:, 1:] = labels[:, :-1]
        shifted.masked_fill_(shifted == -100, self.config.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        encoder_hidden_states=None,
    ):
        if encoder_hidden_states is None:
            if input_ids is None:
                raise ValueError(
                    "input_ids are required when encoder states are not supplied"
                )
            encoder_hidden_states = self.encode(input_ids, attention_mask)
        elif input_ids is not None:
            self._validate_input_ids(
                input_ids, self.config.max_input_length, "input_ids"
            )

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Pass decoder_input_ids or labels to run the decoder"
                )
            decoder_input_ids = self.shift_right(labels)
        if decoder_input_ids.shape[1] > self.config.max_target_length:
            raise ValueError("decoder_input_ids exceed max_target_length")

        decoder_hidden_states = self.decode(
            decoder_input_ids,
            encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )
        if (
            self.config.tie_word_embeddings
            and self.config.scale_decoder_outputs
        ):
            decoder_hidden_states = decoder_hidden_states * (
                self.config.d_model ** -0.5
            )
        logits = self.lm_head(decoder_hidden_states)
        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    "labels must match the decoder logits sequence dimensions"
                )
            loss_labels = labels.masked_fill(
                labels == self.config.pad_token_id, -100
            )
            loss = F.cross_entropy(
                logits.float().reshape(-1, self.config.vocab_size),
                loss_labels.reshape(-1),
                ignore_index=-100,
            )
        return {
            "loss": loss,
            "logits": logits,
            "encoder_hidden_states": encoder_hidden_states,
        }



# Phase-3 uses the encoder representation from this pretrained Phase-1 model.
def _encoder_decoder_encode(self, input_ids, attention_mask=None):
    return self.encode(input_ids, attention_mask)

# The original Phase-1 class already defines encode(); this alias documents the
# task-facing API without changing its state-dict parameter names.

ENCDEC_CONFIG = EncoderDecoderConfig(
    vocab_size=16000,
    d_model=512,
    d_kv=64,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-6,
    relative_attention_num_buckets=32,
    relative_attention_max_distance=128,
    feed_forward_proj="relu",
    dense_act_fn="relu",
    tie_word_embeddings=True,
    scale_decoder_outputs=True,
    initializer_factor=1.0,
    pad_token_id=PAD_ID,
    eos_token_id=EOS_ID,
    decoder_start_token_id=PAD_ID,
    max_input_length=1024,
    max_target_length=1024,
)


def build_encdec_backbone():
    return EncoderDecoderTransformer(ENCDEC_CONFIG)

### 6D. CNN + Causal Self-Attention

Its attention class is named `CNNCausalSelfAttention` so it does not overwrite the decoder's `CausalSelfAttention` (parameter names unchanged).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#1
class CausalConvBlock(nn.Module):


        def __init__(self, dim, kernel_size):
            super().__init__()
        
            self.kernel_size = kernel_size
        
            self.conv = nn.Conv1d(
                in_channels=dim,
                out_channels=dim,
                kernel_size=kernel_size
            )
        
        def forward(self, x):
        
            # x: [B, T, C]
        
            # Step 1: transpose
            x = x.transpose(1, 2)  # [B, C, T]
        
            # Step 2: left padding only
            x = F.pad(x, (self.kernel_size - 1, 0))
        
            # Step 3: convolution
            x = self.conv(x)
        
            # Step 4: transpose back
            x = x.transpose(1, 2)  # [B, T, C]
        
            return x


import math
class CNNCausalSelfAttention(nn.Module):


    def __init__(self, dim, n_heads):
        super().__init__()
    
        assert dim % n_heads == 0
    
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
    
        # Step 1: Linear projections for Q, K, V
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
    
        # Step 2: Final projection
        self.out_proj = nn.Linear(dim, dim)
    
    
    def forward(self, x):
    
        # x: [B, T, C]
        B, T, C = x.shape
    
        # Step 3: Compute Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
    
        # Step 4: Split into heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=True
        )
    
        # Step 10: Merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
    
        # Step 11: Final projection
        out = self.out_proj(out)
    
        return out


class HybridBlock(nn.Module):

        
        def __init__(self, dim, n_heads, kernel_size, ff_mult=4):
            super().__init__()
        
            # Step 1: LayerNorm before CNN
            self.norm1 = nn.LayerNorm(dim)
        
            # Step 2: Causal CNN block
            self.conv = CausalConvBlock(dim, kernel_size)
        
            # Step 3: LayerNorm before Attention
            self.norm2 = nn.LayerNorm(dim)
        
            # Step 4: Causal Self-Attention
            self.attn = CNNCausalSelfAttention(dim, n_heads)
        
            # Step 5: LayerNorm before FFN
            self.norm3 = nn.LayerNorm(dim)
        
            # Step 6: Feed Forward Network
            self.ffn = nn.Sequential(
                nn.Linear(dim, ff_mult * dim),
                nn.GELU(),
                nn.Linear(ff_mult * dim, dim)
            )
        
        
        def forward(self, x):
        
            # x shape: [B, T, C]
        
            # Step 7: CNN + residual
            x = x + self.conv(self.norm1(x))
        
            # Step 8: Attention + residual
            x = x + self.attn(self.norm2(x))
        
            # Step 9: FFN + residual
            x = x + self.ffn(self.norm3(x))
        
            return x



class CNNAttentionLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim,
        n_layers,
        n_heads,
        max_seq_len,
        kernel_size=3,
        ff_mult=4
    ):
        super().__init__()

        # Save important model settings
        self.vocab_size = vocab_size
        self.dim = dim
        self.max_seq_len = max_seq_len

        # Converts each token ID into a dim-dimensional vector
        # Input shape:  [B, T]
        # Output shape: [B, T, dim]
        self.token_emb = nn.Embedding(vocab_size, dim)

        # Learnable positional embeddings
        # Gives the model information about token position in sequence
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Stack multiple CNN + Attention hybrid blocks
        self.blocks = nn.ModuleList([
            HybridBlock(
                dim,
                n_heads,
                kernel_size,
                ff_mult
            )
            for _ in range(n_layers)
        ])

        # Final normalization before vocabulary prediction
        self.final_norm = nn.LayerNorm(dim)

        # Language modeling head
        # Converts hidden representation into logits over vocabulary
        #
        # [B, T, dim] -> [B, T, vocab_size]
        self.lm_head = nn.Linear(dim, vocab_size)


    def forward(self, input_ids, labels=None, **kwargs):

            B, T = input_ids.shape
    
            assert T <= self.max_seq_len
    
            # Token embeddings: [B,T] -> [B,T,C]
            tok = self.token_emb(input_ids)
    
            # Position IDs: [0,1,...,T-1]
            positions = torch.arange(
                T,
                device=input_ids.device
            )
    
            # Position embeddings: [T] -> [T,C]
            pos = self.pos_emb(positions)
    
            # Broadcasting gives [B,T,C]
            x = tok + pos
    
            # CNN + Attention hybrid blocks
            for block in self.blocks:
                x = block(x)
    
            # Final normalization
            x = self.final_norm(x)
    
            # Vocabulary predictions
            # [B,T,C] -> [B,T,V]
            logits = self.lm_head(x)
    
            loss = None
            
            if labels is not None:
            
                # Create targets with the same sequence length
                targets = torch.full_like(
                    labels,
                    -100
                )
            
                # Position t predicts token t+1
                targets[:, :-1] = labels[:, 1:]
            
                # The final position has target -100,
                # so cross entropy ignores it
                loss = F.cross_entropy(
                    logits.reshape(-1, self.vocab_size),
                    targets.reshape(-1),
                    ignore_index=-100
                )
            
                # Trainer only needs loss during training
                return {
                    "loss": loss
                }
            
            # During generation we need logits
            return {
                "logits": logits
            }

# Match the common Phase-1 backbone interface used by Phase 3. The model is
# causal, so right-padding tokens cannot affect earlier sentence token states.
def _cnn_encode(self, input_ids, attention_mask=None):
    batch_size, seq_len = input_ids.shape
    if seq_len > self.max_seq_len:
        raise ValueError(
            f"Sequence length {seq_len} exceeds max_seq_len={self.max_seq_len}"
        )
    token_states = self.token_emb(input_ids)
    position_ids = torch.arange(seq_len, device=input_ids.device)
    hidden_states = token_states + self.pos_emb(position_ids).unsqueeze(0)
    for block in self.blocks:
        hidden_states = block(hidden_states)
    return self.final_norm(hidden_states)

CNNAttentionLM.encode = _cnn_encode
CNN_DIM = 640
CNN_N_LAYERS = 8
CNN_N_HEADS = 8
CNN_MAX_SEQ_LEN = 2048
CNN_KERNEL_SIZE = 3
CNN_FF_MULT = 4

def build_cnn_backbone():
    return CNNAttentionLM(
        vocab_size=16000,
        dim=CNN_DIM,
        n_layers=CNN_N_LAYERS,
        n_heads=CNN_N_HEADS,
        max_seq_len=CNN_MAX_SEQ_LEN,
        kernel_size=CNN_KERNEL_SIZE,
        ff_mult=CNN_FF_MULT,
    )

In [ ]:
# Maximum sequence length accepted by each backbone (chunk sizes must respect it)
BACKBONE_MAX_TOKENS = {
    "decoder": decoder_config.context_length,       # 2048
    "ssm": SSM_MAX_SEQ_LEN,                          # 2048
    "encdec": ENCDEC_CONFIG.max_input_length,        # 1024 (encoder input)
    "cnn": CNN_MAX_SEQ_LEN,                          # 2048
}
print(BACKBONE_MAX_TOKENS)

## 7. Backbone registry and checkpoint checks

`adapter_targets` are the `nn.Linear` module-name suffixes wrapped by LORMA; `ia3_feedforward` must be a subset of them.
For the CNN the feed-forward layers (`HybridBlock.ffn = Sequential(Linear, GELU, Linear)` → `ffn.0`, `ffn.2`) are targeted
together with the attention projections — the configuration of the recorded `cnn lorma` run.
`.safetensors`, `.bin`/`.pt` files and checkpoint directories are accepted.

In [ ]:
BACKBONES = {}


def register_backbone(
    name,
    builder,
    hidden_dim,
    checkpoint_path=None,
    adapter_targets=None,
    ia3_feedforward=None,
):
    BACKBONES[name] = {
        "builder": builder,
        "hidden_dim": hidden_dim,
        "checkpoint_path": checkpoint_path,
        # nn.Linear module-name suffixes for LoRA / rsLoRA / IA3 / LORMA
        "adapter_targets": adapter_targets or [],
        # subset of adapter_targets treated as feed-forward by IA3
        "ia3_feedforward": ia3_feedforward or [],
    }


def _load_state(path):
    from pathlib import Path

    path = Path(path)
    if path.is_dir():
        safe_path = path / "model.safetensors"
        bin_path = path / "pytorch_model.bin"
        if safe_path.is_file():
            path = safe_path
        elif bin_path.is_file():
            path = bin_path
        else:
            raise FileNotFoundError(
                f"No model.safetensors or pytorch_model.bin in {path}"
            )

    if str(path).endswith(".safetensors"):
        from safetensors.torch import load_file
        state = load_file(str(path), device="cpu")
    else:
        try:
            state = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            state = torch.load(path, map_location="cpu")

    if isinstance(state, dict):
        for wrapper_key in ("model_state_dict", "state_dict"):
            if wrapper_key in state and isinstance(state[wrapper_key], dict):
                state = state[wrapper_key]
                break
    return state

def load_backbone(name, checkpoint_path=None, device="cpu", strict=True):
    if name not in BACKBONES:
        raise ValueError(f"Unknown backbone: {name}. Available: {list(BACKBONES)}")

    info = BACKBONES[name]
    model = info["builder"]()

    ckpt = checkpoint_path if checkpoint_path is not None else info["checkpoint_path"]
    if ckpt is None:
        raise ValueError(f"No checkpoint supplied for backbone '{name}'")

    result = model.load_state_dict(_load_state(ckpt), strict=strict)

    print("Backbone:", name, "| checkpoint:", ckpt)
    print("Missing keys   :", result.missing_keys)
    print("Unexpected keys:", result.unexpected_keys)

    return model.to(device)

In [ ]:
register_backbone(
    name="decoder",
    builder=lambda: DecoderOnlyTransformer(decoder_config),
    hidden_dim=decoder_config.n_embd,
    checkpoint_path=DECODER_CHECKPOINT,
    adapter_targets=["qkv", "out_proj", "fc_in", "fc_out"],
    ia3_feedforward=["fc_out"],
)

register_backbone(
    name="ssm",
    builder=build_ssm_backbone,
    hidden_dim=SSM_DIM,
    checkpoint_path=SSM_CHECKPOINT,
    adapter_targets=["in_proj", "out_proj"],
    ia3_feedforward=["out_proj"],
)

register_backbone(
    name="encdec",
    builder=build_encdec_backbone,
    hidden_dim=ENCDEC_CONFIG.d_model,
    checkpoint_path=ENCDEC_CHECKPOINT,
    # wi_0 / wi_1 exist only for gated feed-forward configs; unmatched suffixes are ignored
    adapter_targets=["q", "k", "v", "o", "wi", "wi_0", "wi_1", "wo"],
    ia3_feedforward=["wi", "wi_0", "wi_1", "wo"],
)

register_backbone(
    name="cnn",
    builder=build_cnn_backbone,
    hidden_dim=CNN_DIM,
    checkpoint_path=CNN_CHECKPOINT,
    adapter_targets=["q_proj", "k_proj", "v_proj", "out_proj", "ffn.0", "ffn.2"],
    ia3_feedforward=["ffn.2"],
)

assert set(BACKBONES) == {"decoder", "ssm", "encdec", "cnn"}, list(BACKBONES)
for _name, _info in BACKBONES.items():
    assert set(_info["ia3_feedforward"]) <= set(_info["adapter_targets"]), _name
print("Registered backbones:", list(BACKBONES))

In [ ]:
# Checkpoint presence (strict loading + encode() shapes are tested in section 13).
paths_to_check = {name: info["checkpoint_path"] for name, info in BACKBONES.items()}
paths_to_check["kd_teacher"] = KD_TEACHER_CHECKPOINT
for name, path in paths_to_check.items():
    status = "found" if Path(path).exists() else "MISSING"
    print(f"{name:10s} {status:8s} {path}")

## 8. Phase-5 model wrapper

`chunks → backbone.encode() → chunk vectors [B, C, D] → document pooling → multi-label head → logits [B, NUM_CLASSES]` trained with `BCEWithLogitsLoss`.

For causal backbones (`decoder`, `ssm`, `cnn`) **last-token pooling** is preferred — the final real token has seen the whole chunk.
For the bidirectional `encdec` encoder, **masked mean pooling** is natural. **Attention pooling** adds a learned scorer.
`pool_tokens()` reproduces the three strategies of `Phase3RRModel` so the Phase-5 head can reuse them for token → chunk pooling
(and, with a chunk-level mask, for chunk → document pooling).

In [ ]:
POOLING_STRATEGIES = ("last_token", "mean", "attention")
DEFAULT_POOLING = {"decoder": "last_token", "ssm": "last_token", "cnn": "last_token", "encdec": "mean"}


def pool_tokens(h, mask, strategy, pooler=None):
    """
    h       : [N, L, D] token states,   mask : [N, L] (1 = real token; every row has >= 1 real token)
    strategy: "last_token" | "mean" | "attention" (attention needs pooler = nn.Linear(D, 1, bias=False))
    returns : [N, D]
    """
    if strategy == "last_token":
        last_idx = mask.long().sum(dim=1) - 1
        rows = torch.arange(h.size(0), device=h.device)
        return h[rows, last_idx]
    if strategy == "mean":
        mask_f = mask.unsqueeze(-1).to(h.dtype)
        return (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
    if strategy == "attention":
        assert pooler is not None, "attention pooling needs a learned pooler"
        scores = pooler(h).squeeze(-1)
        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1)
        return (h * weights.unsqueeze(-1)).sum(dim=1)
    raise ValueError(f"Unknown pooling strategy: {strategy}")

In [ ]:
class Phase5LSIModel(nn.Module):
    def __init__(
        self,
        backbone,
        hidden_dim,
        num_classes=None,
        dropout=0.1,
        pos_weight=None,
        pooling_strategy="last_token",
        doc_pooling="mean",
    ):
        super().__init__()

        self.backbone = backbone
        self.hidden_dim = hidden_dim
        self.num_classes = (
            num_classes if num_classes is not None else NUM_CLASSES
        )

        assert pooling_strategy in POOLING_STRATEGIES, pooling_strategy
        assert doc_pooling in {"mean", "max", "attention"}, doc_pooling

        self.pooling_strategy = pooling_strategy
        self.doc_pooling = doc_pooling

        # Trainable token-level attention pooler
        self.chunk_pool = (
            nn.Linear(hidden_dim, 1)
            if pooling_strategy == "attention"
            else None
        )

        # Trainable document-level attention pooler
        self.doc_pool = (
            nn.Linear(hidden_dim, 1)
            if doc_pooling == "attention"
            else None
        )

        self.dropout = nn.Dropout(dropout)

        # Multi-label classification head
        self.classifier = nn.Linear(hidden_dim, self.num_classes)

        # Register pos_weight as a buffer so it follows model.to(device).
        if pos_weight is not None:
            pos_weight = torch.as_tensor(
                pos_weight, dtype=torch.float32
            )
            self.register_buffer("pos_weight", pos_weight)
        else:
            self.register_buffer("pos_weight", None)

        self.loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight
        )

    def head_module_names(self):
        names = ["classifier"]

        if self.chunk_pool is not None:
            names.append("chunk_pool")

        if self.doc_pool is not None:
            names.append("doc_pool")

        return names

    def forward(
        self,
        input_ids,
        attention_mask,
        chunk_mask,
        labels=None,
        has_labels=None,
        **unused,
    ):
        B, C, L = input_ids.shape

        # 1. Flatten documents into individual chunks.
        flat_ids = input_ids.reshape(B * C, L)
        flat_attention_mask = attention_mask.reshape(B * C, L)

        # 2. Encode chunks with the backbone.
        encoded = self.backbone.encode(
            flat_ids,
            attention_mask=flat_attention_mask,
        )

        # Support either a tensor or a HuggingFace-style output.
        if isinstance(encoded, torch.Tensor):
            h = encoded
        elif hasattr(encoded, "last_hidden_state"):
            h = encoded.last_hidden_state
        elif isinstance(encoded, dict):
            h = encoded["last_hidden_state"]
        else:
            raise TypeError(
                "backbone.encode() must return a tensor or "
                "an object/dict containing last_hidden_state"
            )

        # Expected: [B*C, L, D]
        if h.ndim != 3:
            raise ValueError(
                f"Expected encoded states [B*C,L,D], got {h.shape}"
            )

        D = h.shape[-1]
        h = h.reshape(B, C, L, D)

        # 3. Pool tokens into chunk embeddings.
        # pool_tokens is assumed to be defined in Section 6.1.
        chunk_embeddings = pool_tokens(
            h.reshape(B * C, L, D),
            flat_attention_mask,
            self.pooling_strategy,
            pooler=self.chunk_pool,
        )

        chunk_embeddings = chunk_embeddings.reshape(B, C, D)

        # 4. Aggregate real chunks into document embeddings.
        valid_chunks = chunk_mask.bool()

        if self.doc_pooling == "mean":
            weights = valid_chunks.unsqueeze(-1).to(
                chunk_embeddings.dtype
            )
            document_embeddings = (
                (chunk_embeddings * weights).sum(dim=1)
                / weights.sum(dim=1).clamp_min(1.0)
            )

        elif self.doc_pooling == "max":
            masked_embeddings = chunk_embeddings.masked_fill(
                ~valid_chunks.unsqueeze(-1),
                torch.finfo(chunk_embeddings.dtype).min,
            )
            document_embeddings = masked_embeddings.max(dim=1).values

            # Handle documents with no valid chunks defensively.
            has_chunks = valid_chunks.any(dim=1, keepdim=True)
            document_embeddings = torch.where(
                has_chunks,
                document_embeddings,
                torch.zeros_like(document_embeddings),
            )

        elif self.doc_pooling == "attention":
            scores = self.doc_pool(chunk_embeddings).squeeze(-1)
            scores = scores.masked_fill(~valid_chunks, -1e9)

            weights = torch.softmax(scores, dim=1)
            weights = weights * valid_chunks.to(weights.dtype)
            weights = weights / weights.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-9)

            document_embeddings = (
                chunk_embeddings * weights.unsqueeze(-1)
            ).sum(dim=1)

        else:
            raise ValueError(
                f"Unknown document pooling: {self.doc_pooling}"
            )

        # 5. Produce raw multi-label logits.
        logits = self.classifier(
            self.dropout(document_embeddings)
        )

        # 6. Compute loss only on labelled examples.
        loss = None

        if labels is not None:
            if has_labels is None:
                labelled = torch.ones(
                    B, dtype=torch.bool, device=logits.device
                )
            else:
                labelled = has_labels.to(
                    device=logits.device, dtype=torch.bool
                )

            if labelled.any():
                loss = self.loss_fn(
                    logits[labelled],
                    labels.to(
                        device=logits.device,
                        dtype=logits.dtype,
                    )[labelled],
                )

        return {
            "loss": loss,
            "logits": logits,
        }

In [ ]:
def build_phase5_model(backbone_name, device, checkpoint_path=None, backbone=None, **model_kwargs):
    """Generic builder: Phase-1 backbone + Phase-5 classification head."""
    if backbone is None:
        backbone = load_backbone(
            backbone_name,
            checkpoint_path=checkpoint_path,
            device="cpu",
        )

    model = Phase5LSIModel(
        backbone=backbone,
        hidden_dim=BACKBONES[backbone_name]["hidden_dim"],
        **model_kwargs,
    )

    return model.to(device)

## 9. Pooling and class balancing

Token pooling (`pooling_strategy`) and document pooling (`doc_pooling`) are configured per experiment (section 11).
`make_pos_weight` gives each statute a BCE positive weight `(neg/pos)^0.5`, clipped to [1, 20] (`use_pos_weight`).

In [ ]:
def make_pos_weight(cases, statute2idx, power=0.5, max_weight=20.0):
    n = len(cases)
    pos = torch.zeros(len(statute2idx))

    for c in cases:
        for s in set(c["labels"]):
            if s in statute2idx:
                pos[statute2idx[s]] += 1

    neg = n - pos
    w = (neg / pos.clamp_min(1.0)).pow(power)

    return w.clamp(1.0, max_weight)

## 10. Adaptation methods

Three adaptation methods are registered here (`full`, `partial`, `lorma`); knowledge distillation is in section 15.

| key | what is trained | update applied |
|-----|-----------------|----------------|
| `full` | every backbone parameter + head | — |
| `partial` | backbone modules whose names start with a prefix in `cfg.partial_layers` + head | — |
| `lorma` | `A ∈ ℝ^(r×d_out)`, `B ∈ ℝ^(d_out×r)` per targeted `nn.Linear` (`B = 0` at init ⇒ identity) + head | `y = W₀x + b + (α/r)·B A (W₀x)` = `(I + (α/r)·BA)·W₀x + b` (**multiplicative**) |

The Phase-5 head (`model.head_module_names()`: classifier and any attention poolers) is always trainable.

> Checkpoints saved after adaptation contain adapter keys. To reload: **build the model → apply the same adaptation → `load_state_dict`** (use `build_adapted_model` / `load_trained_model`).

In [ ]:
class LowRankMulLinear(nn.Module):
    """LORMA: frozen base nn.Linear + trainable low-rank update (from Phase 2)."""

    def __init__(self, base_layer, rank=16, alpha=32.0):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")

        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features
        device = base_layer.weight.device
        dtype = base_layer.weight.dtype
        #LORA
        # self.A = nn.Parameter(torch.empty(rank, in_dim, device=device, dtype=dtype))
        # self.B = nn.Parameter(torch.zeros(out_dim, rank, device=device, dtype=dtype))
        # nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        #LORMA
        
        self.A = nn.Parameter(
            torch.empty(rank, out_dim, device=device, dtype=dtype)
        )
        
        self.B = nn.Parameter(
            torch.zeros(out_dim, rank, device=device, dtype=dtype)
        )
        
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        #LORA
    # def forward(self, x):
    #     return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

        #LORMA
    def forward(self, x):
        y = self.base(x)
    
        # Apply the multiplicative transformation to W0*x,
        # excluding the frozen bias from the transformation.
        pre = y if self.base.bias is None else y - self.base.bias
    
        update = F.linear(
            F.linear(pre, self.A),
            self.B
        )
    
        return y + self.scale * update


def inject_low_rank(module, target_suffixes, rank, alpha):
    """Replace every nn.Linear whose name ends with a target suffix by LowRankMulLinear. Generic across backbones."""
    to_replace = [
        name for name, m in module.named_modules()
        if isinstance(m, nn.Linear)
        and any(name == t or name.endswith("." + t) for t in target_suffixes)
    ]
    for name in to_replace:
        parent_name, _, child = name.rpartition(".")
        parent = module.get_submodule(parent_name) if parent_name else module
        setattr(parent, child, LowRankMulLinear(getattr(parent, child), rank=rank, alpha=alpha))
    return to_replace

In [ ]:
ADAPTATION_METHODS = {}


def register_adaptation(name):
    def deco(fn):
        ADAPTATION_METHODS[name] = fn
        return fn
    return deco


def _freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def _unfreeze_head(model):
    for name in model.head_module_names():
        module = getattr(model, name)
        for p in module.parameters():
            p.requires_grad = True


def _targets(cfg):
    t = BACKBONES[cfg.backbone_name]["adapter_targets"]
    if not t:
        raise ValueError(
            f"No adapter_targets registered for backbone "
            f"'{cfg.backbone_name}'"
        )
    return t


# ------------------------------------------------------------
# 1. FULL FINE-TUNING
# ------------------------------------------------------------
@register_adaptation("full")
def apply_full(model, cfg):
    # Unfreeze every backbone parameter.
    for p in model.backbone.parameters():
        p.requires_grad = True

    # Ensure every task-head module remains trainable.
    _unfreeze_head(model)

    return model


# ------------------------------------------------------------
# 2. PARTIAL FINE-TUNING
# ------------------------------------------------------------
@register_adaptation("partial")
def apply_partial(model, cfg):
    # Freeze the entire backbone first.
    _freeze_backbone(model)

    # Configure actual module-name prefixes in cfg.partial_layers.
    selected_layers = getattr(cfg, "partial_layers", None)

    if not selected_layers:
        raise ValueError(
            "Partial fine-tuning requires cfg.partial_layers, "
            "a non-empty list of actual backbone module-name prefixes."
        )

    # Inspect registered names rather than assuming decoder conventions.
    module_names = dict(model.backbone.named_modules())
    matched_prefixes = []

    for prefix in selected_layers:
        prefix = prefix.rstrip(".")

        if prefix not in module_names:
            available = [
                name for name in module_names if name
            ]
            raise ValueError(
                f"Partial fine-tuning layer '{prefix}' was not found "
                f"in the backbone. Inspect backbone.named_modules(). "
                f"Available module names include: {available[:30]}"
            )

        matched_prefixes.append(prefix)

    # Unfreeze parameters belonging to selected modules.
    for name, param in model.backbone.named_parameters():
        if any(
            name == prefix or name.startswith(prefix + ".")
            for prefix in matched_prefixes
        ):
            param.requires_grad = True

    # Keep the task-specific head trainable.
    _unfreeze_head(model)

    # Ensure at least one backbone parameter was selected.
    selected_count = sum(
        p.numel()
        for p in model.backbone.parameters()
        if p.requires_grad
    )

    if selected_count == 0:
        raise ValueError(
            "Partial fine-tuning matched no backbone parameters. "
            "Check cfg.partial_layers against named_parameters()."
        )

    print(f"Partial fine-tuning: {selected_count:,} backbone parameters unfrozen")
    return model


# ------------------------------------------------------------
# 3. LORMA (UNCHANGED)
# ------------------------------------------------------------
@register_adaptation("lorma")
def apply_lorma(model, cfg):
    _freeze_backbone(model)

    replaced = inject_low_rank(
        model.backbone,
        target_suffixes=_targets(cfg),
        rank=cfg.lorma_rank,
        alpha=cfg.lorma_alpha,
    )

    if not replaced:
        raise RuntimeError(
            f"LORMA: No matching linear layers found for {_targets(cfg)}"
        )

    print(f"LORMA: wrapped {len(replaced)} Linear layers")
    _unfreeze_head(model)
    return model


# ------------------------------------------------------------
# APPLY ADAPTATION
# ------------------------------------------------------------
def apply_adaptation(model, cfg):
    if cfg.adaptation_method not in ADAPTATION_METHODS:
        raise ValueError(
            f"Unknown adaptation '{cfg.adaptation_method}'. "
            f"Available: {list(ADAPTATION_METHODS)}"
        )

    model = ADAPTATION_METHODS[cfg.adaptation_method](model, cfg)
    report_trainable(model)
    return model


def report_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    print(f"Total params    : {total:,}")
    print(
        f"Trainable params: {trainable:,} "
        f"({100 * trainable / max(total, 1):.3f}%)"
    )

    return total, trainable


# ------------------------------------------------------------
# REGISTRY VALIDATION
# ------------------------------------------------------------
assert set(ADAPTATION_METHODS) == {
    "full", "partial", "lorma"
}, ADAPTATION_METHODS.keys()

print("Adaptation methods:", list(ADAPTATION_METHODS))

## 11. Experiment configuration

`exp_cfg` is trained by section 14 when `RUN_TRAINING = True`. The active configuration is the best recorded run
(`cnn lorma`: CNN backbone, LORMA on attention + feed-forward layers, mean token pooling, no `pos_weight`).
`partial` needs `partial_layers` (backbone module-name prefixes, e.g. `["blocks.6", "blocks.7", "final_ln"]`).

In [ ]:
@dataclass
class ExperimentConfig:
    backbone_name: str
    adaptation_method: str          # "full" | "partial" | "lorma"

    learning_rate: float            # adapter / backbone-side LR
    batch_size: int
    grad_accum: int
    epochs: int

    head_learning_rate: float = 3e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool = True
    dropout: float = 0.1
    pooling_strategy: str = "last_token"   # token -> chunk: "last_token" | "mean" | "attention"
    doc_pooling: str = "mean"             # chunk -> document aggregation
    use_pos_weight: bool = False

    # Multi-label decision rule: sigmoid(logit) >= prediction_threshold
    prediction_threshold: float = 0.5
    partial_layers: Optional[List[str]] = None
    # LORMA
    lorma_rank: int = 16
    lorma_alpha: float = 32.0

    seed: int = SEED
    run_name: Optional[str] = None

    @property
    def run_dir(self):
        name = self.run_name or f"phase5_{self.backbone_name}_{self.adaptation_method}"
        return WORK_DIR / name


def save_experiment_config(cfg, run_dir=None):
    run_dir = Path(run_dir or cfg.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)


# Best recorded run (Codabench 0.2846): CNN + LORMA, mean token pooling.
exp_cfg = ExperimentConfig(
    backbone_name="cnn",
    adaptation_method="lorma",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=8,
    lorma_rank=16,
    lorma_alpha=32.0,
    pooling_strategy="mean",
    prediction_threshold=0.5,
    run_name="phase5_cnn_lorma",
)
# Decoder + LORMA:
# exp_cfg = ExperimentConfig(
#     backbone_name="decoder",
#     adaptation_method="lorma",
#     learning_rate=5e-4,
#     batch_size=BATCH_SIZE,
#     grad_accum=8,
#     epochs=8,
#     lorma_rank=16,
#     lorma_alpha=32.0,
#     pooling_strategy="last_token",
#     prediction_threshold=0.5,
# )
assert CHUNK_TOKENS <= BACKBONE_MAX_TOKENS[exp_cfg.backbone_name], "chunk longer than the backbone accepts (encdec: 1024)"


assert exp_cfg.backbone_name in BACKBONES, exp_cfg.backbone_name
assert exp_cfg.adaptation_method in ADAPTATION_METHODS, exp_cfg.adaptation_method
print(asdict(exp_cfg))

In [ ]:
def build_adapted_model(cfg, device=DEVICE, backbone=None, **extra_model_kwargs):
    """Phase-1 backbone + Phase-5 head + chosen adaptation. Use this both for training and for reloading checkpoints."""
    set_seed(cfg.seed)
    model = build_phase5_model(
        backbone_name=cfg.backbone_name,
        device="cpu",
        backbone=backbone,
        dropout=cfg.dropout,
        pooling_strategy=cfg.pooling_strategy,
        doc_pooling=cfg.doc_pooling,
        num_classes=NUM_CLASSES,
        **extra_model_kwargs,
    )
    model = apply_adaptation(model, cfg)
    return model.to(device)


def build_optimizer(model, cfg):
    head_names = tuple(f"{n}." for n in model.head_module_names())
    head_params, adapter_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith(head_names) else adapter_params).append(p)

    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": cfg.learning_rate})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_learning_rate})
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, cfg, steps_per_epoch):
    total_steps = max(math.ceil(steps_per_epoch / cfg.grad_accum) * cfg.epochs, 1)
    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[g["lr"] for g in optimizer.param_groups],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy="cos",
    )


def pos_weight_for(cfg):
    """BCE pos_weight exactly as used in training for this configuration."""
    return make_pos_weight(train_cases, statute2idx) if cfg.use_pos_weight else None


def load_trained_model(cfg, checkpoint_path=None, backbone=None):
    """Rebuild backbone + head + adaptation for `cfg`, then load a trained Phase-5 checkpoint strictly."""
    path = Path(checkpoint_path or (cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"Trained checkpoint not found: {path}")
    model = build_adapted_model(cfg, backbone=backbone, pos_weight=pos_weight_for(cfg))
    state = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.eval()
    print("Loaded trained model:", path)
    return model

## 12. Training, evaluation and submission utilities

Definitions only — nothing in this section trains a model.

Generic loops: every tensor in the batch is moved to the device and passed to `model(**batch)`; non-tensor fields (`ids`) are kept aside.

In [ ]:
def split_batch(batch, device):
    tensors, meta = {}, {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            tensors[k] = v.to(device, non_blocking=True)
        else:
            meta[k] = v
    return tensors, meta


def _amp_enabled(use_amp):
    return use_amp and torch.cuda.is_available()


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
    scaler=None,
    scheduler=None,
):
    model.train()
    amp = _amp_enabled(use_amp)

    if scaler is None:
        scaler = make_scaler(amp)

    assert grad_accum >= 1

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        tensors, _ = split_batch(batch, device)

        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=amp
        ):
            out = model(**tensors)
            raw_loss = out["loss"]
            if raw_loss is None:
                raise RuntimeError("Training batch has no labelled examples")
            loss = raw_loss / current_accum

        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                f"Non-finite training loss at batch {step}"
            )

        scaler.scale(loss).backward()

        update_now = (
            (step + 1) % grad_accum == 0
            or (step + 1) == num_batches
        )

        if update_now:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    max_grad_norm
                )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            step_succeeded = (
                not amp or scaler.get_scale() >= old_scale
            )

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None and step_succeeded:
                scheduler.step()

        total_loss += raw_loss.detach().item()

    return total_loss / max(num_batches, 1)


@torch.no_grad()
def run_validation(model, loader, device, use_amp=True):
    """
    Returns:
        loss   : mean loss over labelled batches (None for test)
        logits : Tensor [N, NUM_CLASSES] (cpu, float32), loader order
        labels : Tensor [N, NUM_CLASSES] multi-hot
        ids    : List[str]
    """
    model.eval()
    amp = _amp_enabled(use_amp)

    total_loss, n_labelled = 0.0, 0
    all_logits, all_labels, all_ids = [], [], []

    for batch in tqdm(loader, desc="eval", leave=False):
        tensors, meta = split_batch(batch, device)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = model(**tensors)

        if out.get("loss") is not None and torch.isfinite(out["loss"]):
            n = int(tensors["has_labels"].sum().item())
            total_loss += out["loss"].item() * n
            n_labelled += n

        all_logits.append(out["logits"].float().cpu())
        all_labels.append(tensors["labels"].float().cpu())
        all_ids.extend(meta["ids"])

    return {
        "loss": total_loss / n_labelled if n_labelled else None,
        "logits": torch.cat(all_logits),
        "labels": torch.cat(all_labels),
        "ids": all_ids,
    }

### 12.1 Metrics and thresholds

Macro-F1 (primary), Micro-F1 and Subset Accuracy (exact match of the predicted statute set), plus the sigmoid-threshold decision rule
that turns logits into a set of discrete statute IDs.

Dev unseen-in-train statutes cannot be predicted; keep them in mind when comparing local Macro-F1 to Codabench.

In [ ]:
import torch
import numpy as np


def logits_to_multi_hot(logits, threshold=0.5, min_labels=0):
    """
    logits    : Tensor [N, K]
    threshold : float or Tensor [K]
    min_labels: fallback to top-k only for rows with no predictions
    returns   : LongTensor [N, K]
    """
    probs = torch.sigmoid(logits)

    threshold = torch.as_tensor(
        threshold,
        dtype=probs.dtype,
        device=probs.device,
    )

    pred = probs >= threshold

    if min_labels > 0:
        empty_rows = ~pred.any(dim=1)

        if empty_rows.any():
            k = min(min_labels, probs.shape[1])
            top_indices = probs[empty_rows].topk(k, dim=1).indices

            rows = torch.where(empty_rows)[0]
            pred[rows.unsqueeze(1), top_indices] = True

    return pred.long()


def multi_hot_to_statute_ids(row, id2statute):
    """
    Convert one multi-hot row to sorted, unique Python-int statute IDs.
    """
    if isinstance(row, torch.Tensor):
        row = row.detach().cpu().flatten().tolist()
    elif isinstance(row, np.ndarray):
        row = row.flatten().tolist()

    statute_ids = {
        int(id2statute[idx])
        for idx, active in enumerate(row)
        if active
    }

    return sorted(statute_ids)


def compute_multilabel_metrics(gold, pred):
    """
    Compute multilabel metrics over every class in the vocabulary.

    Macro metrics include all classes, assigning zero to undefined
    precision/recall/F1 when the corresponding denominator is zero.
    """
    from sklearn.metrics import (
        f1_score,
        precision_score,
        recall_score,
        accuracy_score,
    )

    if isinstance(gold, torch.Tensor):
        gold = gold.detach().cpu().numpy()

    if isinstance(pred, torch.Tensor):
        pred = pred.detach().cpu().numpy()

    gold = np.asarray(gold).astype(bool)
    pred = np.asarray(pred).astype(bool)

    if gold.shape != pred.shape:
        raise ValueError(
            f"Shape mismatch: gold={gold.shape}, pred={pred.shape}"
        )

    if gold.ndim != 2:
        raise ValueError("gold and pred must have shape [N, K]")

    if gold.shape[0] == 0:
        raise ValueError("Cannot calculate metrics on an empty dataset")

    tp = np.logical_and(gold, pred).sum(axis=0).astype(np.float64)
    fp = np.logical_and(~gold, pred).sum(axis=0).astype(np.float64)
    fn = np.logical_and(gold, ~pred).sum(axis=0).astype(np.float64)

    precision = np.divide(
        tp, tp + fp,
        out=np.zeros_like(tp),
        where=(tp + fp) > 0,
    )

    recall = np.divide(
        tp, tp + fn,
        out=np.zeros_like(tp),
        where=(tp + fn) > 0,
    )

    f1 = np.divide(
        2 * tp, 2 * tp + fp + fn,
        out=np.zeros_like(tp),
        where=(2 * tp + fp + fn) > 0,
    )

    support = gold.sum(axis=0).astype(int)

    per_class = {
        int(id2statute[idx]): {
            "precision": float(precision[idx]),
            "recall": float(recall[idx]),
            "f1": float(f1[idx]),
            "support": int(support[idx]),
        }
        for idx in range(gold.shape[1])
    }

    total_tp = tp.sum()
    total_fp = fp.sum()
    total_fn = fn.sum()

    micro_precision = (
        float(total_tp / (total_tp + total_fp))
        if total_tp + total_fp > 0 else 0.0
    )

    micro_recall = (
        float(total_tp / (total_tp + total_fn))
        if total_tp + total_fn > 0 else 0.0
    )

    micro_f1 = (
        float(2 * total_tp / (2 * total_tp + total_fp + total_fn))
        if 2 * total_tp + total_fp + total_fn > 0 else 0.0
    )

    # Cross-check with sklearn; all K classes are included.
    assert np.isclose(
        f1_score(gold, pred, average="macro", zero_division=0),
        f1.mean(),
    )
    assert np.isclose(
        f1_score(gold, pred, average="micro", zero_division=0),
        micro_f1,
    )
    
    assert np.isclose(
        accuracy_score(gold, pred),
        np.mean(np.all(gold == pred, axis=1)),
    )
    return {
        "macro_f1": float(f1.mean()),
        "micro_f1": micro_f1,
        "subset_accuracy": float(accuracy_score(gold, pred)),
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "per_class": per_class,
    }


def tune_threshold(logits, gold, grid=None, per_class=False):
    """
    Tune thresholds to maximise development Macro-F1.

    per_class=False: returns a scalar float.
    per_class=True : returns a Tensor [K].
    """
    if isinstance(logits, torch.Tensor):
        logits = logits.detach().cpu()
    else:
        logits = torch.as_tensor(logits)

    if isinstance(gold, torch.Tensor):
        gold = gold.detach().cpu()
    else:
        gold = torch.as_tensor(gold)

    gold = gold.bool()

    if logits.ndim != 2 or gold.shape != logits.shape:
        raise ValueError("logits and gold must have matching [N, K] shapes")

    if logits.shape[0] == 0:
        raise ValueError("Cannot tune thresholds on empty dev data")

    if grid is None:
        grid = [i / 100 for i in range(5, 96, 5)]

    grid = sorted(set(float(t) for t in grid))

    if not grid or any(t < 0 or t > 1 for t in grid):
        raise ValueError("grid must contain thresholds in [0, 1]")

    probs = torch.sigmoid(logits)
    gold_np = gold.numpy().astype(bool)

    # First tune a global threshold.
    best_global = grid[0]
    best_global_f1 = -1.0

    for threshold in grid:
        pred = (probs >= threshold).numpy()
        score = compute_multilabel_metrics(
            gold_np, pred
        )["macro_f1"]

        if score > best_global_f1:
            best_global_f1 = score
            best_global = threshold

    if not per_class:
        return best_global

    # Tune each class independently, falling back to global threshold
    # for classes with too few positive dev examples.
    K = logits.shape[1]
    thresholds = torch.full((K,), best_global, dtype=torch.float32)

    min_positives = 5

    for k in range(K):
        positive_count = int(gold[:, k].sum())

        if positive_count < min_positives:
            continue

        best_threshold = best_global
        best_f1 = -1.0

        for threshold in grid:
            pred_k = probs[:, k] >= threshold
            gold_k = gold[:, k]

            tp = int((pred_k & gold_k).sum())
            fp = int((pred_k & ~gold_k).sum())
            fn = int((~pred_k & gold_k).sum())

            denominator = 2 * tp + fp + fn
            score = 2 * tp / denominator if denominator else 0.0

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds[k] = best_threshold

    return thresholds

### 12.2 Training loop

In [ ]:
def fit(cfg, train_loader, dev_loader):
    run_dir = cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(cfg)

    pos_weight = pos_weight_for(cfg)
    model = build_adapted_model(cfg, pos_weight=pos_weight)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, len(train_loader))
    scaler = make_scaler(_amp_enabled(cfg.use_amp))

    best_f1, history = -1.0, []

    for epoch in range(cfg.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, DEVICE,
            grad_accum=cfg.grad_accum, max_grad_norm=cfg.max_grad_norm,
            use_amp=cfg.use_amp, scaler=scaler, scheduler=scheduler,
        )
        val_out = run_validation(model, dev_loader, DEVICE, use_amp=cfg.use_amp)
        pred = logits_to_multi_hot(val_out["logits"], cfg.prediction_threshold)    # Section 10
        metrics = compute_multilabel_metrics(val_out["labels"], pred)              # Section 10

        row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_out["loss"],
               **{k: metrics[k] for k in ("macro_f1", "micro_f1", "subset_accuracy")},
               "time_s": round(time.time() - t0, 1)}
        history.append(row)
        print(row)

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]
            torch.save(model.state_dict(), run_dir / "best_model.pt")
            with open(run_dir / "metrics.json", "w") as f:
                json.dump({**metrics, "threshold": cfg.prediction_threshold}, f, indent=2)
            print(f"  ✓ new best Macro-F1 {best_f1:.4f}")

    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # reload the best checkpoint (as in Phase 3) so the returned model is the selected one
    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location=DEVICE, weights_only=True))
    print(f"Training complete. Best: {best_f1:.4f}")
    return model, best_f1, history

In [ ]:
def tiny_overfit_test(cfg, n_items=4, epochs=30):
    model = build_adapted_model(cfg)
    tiny_loader = DataLoader(
        Subset(train_ds, range(min(n_items, len(train_ds)))),
        batch_size=2, shuffle=True, collate_fn=lsi_collator,
    )
    opt = build_optimizer(model, replace(cfg, learning_rate=cfg.learning_rate * 5))
    for i in range(epochs):
        loss = train_one_epoch(model, tiny_loader, opt, DEVICE, grad_accum=1, use_amp=cfg.use_amp)
        if i % 5 == 0 or i == epochs - 1:
            print(f"step {i:3d} loss {loss:.4f}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### 12.3 Test inference and submission helpers

Expected `prediction.json` — a JSON **list**, one object per test case; `pred` = unique integer statute IDs:

```json
[
  {"id": "case_001", "pred": [1, 7, 23]},
  {"id": "case_002", "pred": [4]}
]
```

Use the threshold tuned on **dev** (section 16) and generate predictions from a **freshly loaded** best checkpoint.

In [ ]:
def run_inference(
    model,
    loader,
    device,
    use_amp,
    threshold=0.5,
    min_labels=0,
    id2statute=None,
):
    """
    Returns:
        [
            {"id": original_case_id, "pred": [sorted statute IDs]},
            ...
        ]
    """
    if id2statute is None:
        id2statute = globals().get("id2statute")

    if id2statute is None:
        raise ValueError("id2statute mapping must be provided")

    out = run_validation(model, loader, device, use_amp)

    multi_hot = logits_to_multi_hot(
        out["logits"],
        threshold=threshold,
        min_labels=min_labels,
    )
    assert len(out["ids"]) == len(multi_hot), "ID/prediction count mismatch"
    predictions = []

    for case_id, prediction_row in zip(out["ids"], multi_hot):
        statute_ids = multi_hot_to_statute_ids(
            prediction_row,
            id2statute,
        )

        predictions.append({
            "id": case_id,
            "pred": statute_ids,
        })

    if len(predictions) != len(out["ids"]):
        raise RuntimeError("Prediction count does not match case ID count")

    return predictions


def validate_submission_format(
    predictions,
    official_test_ids,
    valid_statutes=None,
):
    """
    Validate submission structure, IDs, and statute IDs.

    Raises ValueError if validation fails.
    Returns True if valid.
    """
    if not isinstance(predictions, list):
        raise ValueError("predictions must be a list")

    # Official IDs must themselves be unique.
    official_test_ids = list(official_test_ids)

    if len(official_test_ids) != len(set(official_test_ids)):
        raise ValueError("official_test_ids contains duplicate IDs")

    expected_ids = set(official_test_ids)
    seen_ids = set()

    valid_statutes_set = (
        set(valid_statutes) if valid_statutes is not None else None
    )

    for i, entry in enumerate(predictions):
        if not isinstance(entry, dict):
            raise ValueError(f"Prediction {i} must be a dictionary")

        if set(entry.keys()) != {"id", "pred"}:
            raise ValueError(
                f"Prediction {i} must contain exactly 'id' and 'pred'"
            )

        case_id = entry["id"]

        if case_id not in expected_ids:
            raise ValueError(
                f"Unexpected test ID at prediction {i}: {case_id!r}"
            )

        if case_id in seen_ids:
            raise ValueError(f"Duplicate test ID: {case_id!r}")

        seen_ids.add(case_id)

        pred = entry["pred"]

        if not isinstance(pred, list):
            raise ValueError(
                f"Prediction for ID {case_id!r} must be a list"
            )

        for statute_id in pred:
            if type(statute_id) is not int:
                raise ValueError(
                    f"Statute IDs must be Python ints; got "
                    f"{statute_id!r} for ID {case_id!r}"
                )

        if len(pred) != len(set(pred)):
            raise ValueError(
                f"Duplicate statute IDs for case {case_id!r}"
            )

        if valid_statutes_set is not None:
            invalid = set(pred) - valid_statutes_set

            if invalid:
                raise ValueError(
                    f"Invalid statute IDs for case {case_id!r}: "
                    f"{sorted(invalid)}"
                )

    missing_ids = expected_ids - seen_ids

    if missing_ids:
        raise ValueError(
            f"Missing {len(missing_ids)} official test IDs; "
            f"examples: {list(missing_ids)[:10]}"
        )

    if len(predictions) != len(official_test_ids):
        raise ValueError(
            f"Expected {len(official_test_ids)} predictions, "
            f"got {len(predictions)}"
        )

    print(f"Submission valid: {len(predictions)} predictions")
    return True

In [ ]:
def save_submission(predictions, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_path = output_dir / "prediction.json"
    zip_path = output_dir / "submission.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    print("Saved:", pred_path)
    print("Saved:", zip_path)
    return pred_path, zip_path


def verify_zip_root(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at root, got {names}"
        obj = json.loads(z.read("prediction.json").decode("utf-8"))
    print("Zip root OK:", names, "| entries:", len(obj))
    return obj

In [ ]:
def predict_test(model, cfg, per_class_threshold=False):
    """Tune the decision threshold on dev, then predict the test set with it."""
    dev_out = run_validation(model, dev_loader, DEVICE, use_amp=cfg.use_amp)
    threshold = tune_threshold(dev_out["logits"], dev_out["labels"], per_class=per_class_threshold)
    dev_metrics = compute_multilabel_metrics(dev_out["labels"], logits_to_multi_hot(dev_out["logits"], threshold))
    print("Dev-tuned threshold:", threshold if not torch.is_tensor(threshold) else "per-class",
          "| dev Macro-F1:", round(dev_metrics["macro_f1"], 4), "| Micro-F1:", round(dev_metrics["micro_f1"], 4))
    preds = run_inference(model, test_loader, device=DEVICE, use_amp=cfg.use_amp,
                          threshold=threshold, id2statute=id2statute)
    return preds, threshold, dev_metrics

## 13. Preflight tests (optional, lightweight)

Run when `RUN_PREFLIGHT = True`. No full training. Every check is reported as PASS/FAIL; a missing checkpoint is
reported with its exact path (never skipped), and the summary cell raises if anything failed.

1. Every backbone in `PREFLIGHT_BACKBONES` loads strictly from its checkpoint.
2. Every `encode()` returns `[B, L, D]`.
3. Dataset and collator: tensor shapes, chunk / padding masks, multi-hot targets, first-`MAX_CHUNKS` selection, test targets empty.
4. The Phase-5 wrapper produces `[B, NUM_CLASSES]` logits and a finite loss for every token-pooling strategy.
5. Padded chunk slots do not change a case's logits.
6. Every adaptation method trains exactly the intended parameters (all backbones; `partial` on the decoder).
7. A small batch supports forward, loss and backward with gradients only on trainable parameters.
8. Inference without labels works (`loss` is `None`).
9. Threshold tuning, test predictions and submission files validate.

In [ ]:
PREFLIGHT_RESULTS = []


def preflight_check(name, fn):
    try:
        detail = fn()
        PREFLIGHT_RESULTS.append((name, "PASS", detail or ""))
        print(f"[PASS] {name}" + (f" - {detail}" if detail else ""))
    except Exception as exc:
        PREFLIGHT_RESULTS.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
        print(f"[FAIL] {name} - {type(exc).__name__}: {exc}")


def _fail(msg):
    raise RuntimeError(msg)


def small_batch(ds, n_items=2, n_chunks=3, n_tok=24):
    """Tiny collated batch with different chunk counts per case (exercises chunk padding)."""
    items = []
    for k in range(min(n_items, len(ds))):
        it = ds[k]
        keep = max(1, n_chunks - k)
        chunks = [c if len(c) <= n_tok else c[: n_tok - 1] + [EOS_ID] for c in it["chunks"][:keep]]
        items.append({"id": it["id"], "chunks": chunks, "target": it["target"], "has_labels": it["has_labels"]})
    return lsi_collator(items)


def is_adapter_param(name):
    return name.endswith(".A") or name.endswith(".B")

In [ ]:
# ---- Tests 1, 2, 6: every backbone -> strict load, encode() shape, every adaptation method ----
pf_backbone = None

if RUN_PREFLIGHT:
    probe_ids = torch.tensor(
        [[BOS_ID] + tokenizer.encode("The appeal is dismissed with costs.") + [EOS_ID]] * 2
    )

    for name in PREFLIGHT_BACKBONES:
        info = BACKBONES[name]
        ckpt = info["checkpoint_path"]
        loaded = {}

        def _load():
            if not Path(ckpt).exists():
                raise FileNotFoundError(f"missing checkpoint: {ckpt}")
            bb = load_backbone(name, device="cpu", strict=True)
            loaded["bb"] = bb
            return f"strict load OK ({sum(p.numel() for p in bb.parameters()):,} params)"

        preflight_check(f"1. {name}: checkpoint loads strictly", _load)
        if "bb" not in loaded:
            preflight_check(f"2. {name}: encode() shape", lambda: _fail("backbone not loaded"))
            continue
        bb = loaded["bb"]

        def _encode():
            bb.eval()
            with torch.no_grad():
                h = bb.encode(probe_ids, torch.ones_like(probe_ids))
            expected = (probe_ids.shape[0], probe_ids.shape[1], info["hidden_dim"])
            assert tuple(h.shape) == expected, (tuple(h.shape), expected)
            assert torch.isfinite(h).all()
            assert CHUNK_TOKENS <= BACKBONE_MAX_TOKENS[name]
            return f"{tuple(h.shape)}"

        preflight_check(f"2. {name}: encode() returns [B, L, D]", _encode)

        methods = ["full", "lorma"] + (["partial"] if name == "decoder" else [])
        for method in methods:
            def _adapt(method=method):
                cfg = replace(exp_cfg, backbone_name=name, adaptation_method=method,
                              partial_layers=["blocks.6", "blocks.7", "final_ln"] if method == "partial" else None)
                m = Phase5LSIModel(copy.deepcopy(bb), info["hidden_dim"], num_classes=NUM_CLASSES)
                m = ADAPTATION_METHODS[method](m, cfg)
                bb_params = list(m.backbone.named_parameters())
                trainable = [n for n, p in bb_params if p.requires_grad]
                head_frozen = [n for n, p in m.classifier.named_parameters() if not p.requires_grad]
                assert not head_frozen, f"frozen head parameters: {head_frozen}"
                if method == "full":
                    assert len(trainable) == len(bb_params), "full fine-tuning left backbone parameters frozen"
                elif method == "partial":
                    wrong = [n for n in trainable if not n.startswith(("blocks.6.", "blocks.7.", "final_ln."))]
                    assert trainable and not wrong, f"unexpected trainable parameters: {wrong[:3]}"
                else:
                    assert trainable, "no adapter parameters are trainable"
                    wrong = [n for n in trainable if not is_adapter_param(n)]
                    assert not wrong, f"base backbone parameters left trainable: {wrong[:3]}"
                n_bb = sum(p.numel() for _, p in bb_params if p.requires_grad)
                del m
                return f"trainable backbone params {n_bb:,}"

            preflight_check(f"6. {name}: {method} trainable parameters", _adapt)

        if name == exp_cfg.backbone_name:
            pf_backbone = bb          # reused below; avoids loading the checkpoint again
        else:
            del bb
        loaded.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("RUN_PREFLIGHT=False: preflight skipped.")

In [ ]:
# ---- Tests 3, 4, 5, 7, 8: data pipeline and the Phase-5 wrapper on exp_cfg's backbone ----
if RUN_PREFLIGHT:

    def _collator():
        b = next(iter(dev_loader))
        B, C, L = b["input_ids"].shape
        assert b["attention_mask"].shape == (B, C, L)
        assert b["chunk_mask"].shape == (B, C) and b["chunk_mask"].dtype == torch.bool
        assert b["labels"].shape == (B, NUM_CLASSES) and b["labels"].dtype == torch.float32
        assert b["has_labels"].all() and (b["labels"].sum(1) >= 1).all()
        assert (b["attention_mask"].sum(-1) >= 1).all(), "all-masked chunk row"
        assert (b["attention_mask"][b["chunk_mask"]].sum(-1) >= 2).all(), "real chunk without BOS/EOS"
        assert C <= MAX_CHUNKS and L <= CHUNK_TOKENS
        tb = next(iter(test_loader))
        assert not tb["has_labels"].any() and (tb["labels"] == 0).all(), "test targets must be empty"
        longest = max(dev_cases, key=lambda c: len(c["text"]))
        full = chunk_token_ids(tokenizer.encode(longest["text"]), CHUNK_TOKENS, CHUNK_STRIDE)
        item = next(it for it in dev_ds.items if it["id"] == longest["id"])
        assert item["chunks"] == full[:MAX_CHUNKS], "LSI_Dataset must keep the first MAX_CHUNKS chunks"
        gold = sorted(id2statute[i] for i in item["target"].nonzero().flatten().tolist())
        assert gold == longest["labels"], "multi-hot target does not match the statute list"
        return f"dev batch {(B, C, L)}; longest dev case {len(full)} chunks -> {len(item['chunks'])} kept"

    preflight_check("3. dataset / collator / chunk selection", _collator)

    if pf_backbone is None:
        preflight_check("4-8. model wrapper tests",
                        lambda: _fail(f"backbone '{exp_cfg.backbone_name}' was not loaded in the previous cell"))
    else:
        hidden = BACKBONES[exp_cfg.backbone_name]["hidden_dim"]
        pf_batch = small_batch(dev_ds)
        pf_tensors = {k: v for k, v in pf_batch.items() if torch.is_tensor(v)}
        B = pf_tensors["input_ids"].shape[0]

        def _forward():
            done = []
            for pooling in POOLING_STRATEGIES:
                m = Phase5LSIModel(pf_backbone, hidden, num_classes=NUM_CLASSES, pooling_strategy=pooling,
                                   doc_pooling=exp_cfg.doc_pooling).eval()
                with torch.no_grad():
                    out = m(**pf_tensors)
                assert tuple(out["logits"].shape) == (B, NUM_CLASSES), out["logits"].shape
                assert torch.isfinite(out["loss"])
                done.append(pooling)
            return f"{', '.join(done)} -> {(B, NUM_CLASSES)}"

        preflight_check("4. wrapper forward -> [B, NUM_CLASSES] + finite loss", _forward)

        def _padding():
            m = Phase5LSIModel(pf_backbone, hidden, num_classes=NUM_CLASSES, pooling_strategy=exp_cfg.pooling_strategy,
                               doc_pooling=exp_cfg.doc_pooling).eval()
            n1 = int(pf_batch["chunk_mask"][1].sum())
            alone = lsi_collator([{
                "id": "x",
                "chunks": [pf_batch["input_ids"][1, j][pf_batch["attention_mask"][1, j].bool()].tolist() for j in range(n1)],
                "target": pf_batch["labels"][1],
                "has_labels": True,
            }])
            with torch.no_grad():
                batched = m(**pf_tensors)["logits"][1]
                single = m(**{k: v for k, v in alone.items() if torch.is_tensor(v)})["logits"][0]
            diff = (batched - single).abs().max().item()
            assert diff < 1e-3, f"chunk padding changed logits by {diff:.2e}"
            return f"max diff {diff:.1e}"

        preflight_check("5. padded chunks do not change logits", _padding)

        def _backward():
            m = Phase5LSIModel(copy.deepcopy(pf_backbone), hidden, num_classes=NUM_CLASSES,
                               pooling_strategy=exp_cfg.pooling_strategy, doc_pooling=exp_cfg.doc_pooling,
                               pos_weight=pos_weight_for(exp_cfg))
            m = ADAPTATION_METHODS[exp_cfg.adaptation_method](m, exp_cfg).to(DEVICE).train()
            out = m(**{k: v.to(DEVICE) for k, v in pf_tensors.items()})
            out["loss"].backward()
            no_grad = [n for n, p in m.named_parameters() if p.requires_grad and p.grad is None]
            leaked = [n for n, p in m.named_parameters() if not p.requires_grad and p.grad is not None]
            assert not no_grad, f"trainable parameters without gradient: {no_grad[:3]}"
            assert not leaked, f"frozen parameters received gradients: {leaked[:3]}"
            n = sum(1 for p in m.parameters() if p.requires_grad)
            del m, out
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return f"{exp_cfg.adaptation_method}: {n} trainable tensors all received gradients"

        preflight_check("7. forward + loss + backward on a small batch", _backward)

        def _no_labels():
            m = Phase5LSIModel(pf_backbone, hidden, num_classes=NUM_CLASSES,
                               pooling_strategy=exp_cfg.pooling_strategy).eval()
            with torch.no_grad():
                out = m(**{k: v for k, v in pf_tensors.items() if k not in ("labels", "has_labels")})
            assert out["loss"] is None
            assert tuple(out["logits"].shape) == (B, NUM_CLASSES)
            return "loss=None, logits OK"

        preflight_check("8. inference without labels", _no_labels)

In [ ]:
# ---- Test 9: threshold tuning, test predictions and submission files (dummy model, no backbone) ----
if RUN_PREFLIGHT:

    class _PriorModel(nn.Module):
        """Predicts every statute's training prior (no backbone)."""
        def __init__(self):
            super().__init__()
            prior = torch.stack([ds_item["target"] for ds_item in train_ds.items]).mean(0).clamp(1e-4, 1 - 1e-4)
            self.register_buffer("logit", torch.log(prior / (1 - prior)))

        def forward(self, input_ids, attention_mask, chunk_mask, labels=None, has_labels=None, **unused):
            return {"loss": None, "logits": self.logit.expand(input_ids.shape[0], -1).clone()}

    def _submission():
        _cfg = replace(exp_cfg, use_amp=False)
        preds, threshold, _ = predict_test(_PriorModel(), _cfg)
        validate_submission_format(preds, official_test_ids=[c["id"] for c in test_cases], valid_statutes=set(STATUTE_IDS))
        assert [p["id"] for p in preds] == [c["id"] for c in test_cases], "test order changed"
        _, zip_path = save_submission(preds, WORK_DIR / "_preflight_submission")
        verify_zip_root(zip_path)
        return f"{len(preds)} test cases, threshold {threshold}"

    preflight_check("9. threshold tuning, test predictions and submission validation", _submission)

In [ ]:
# ---- Preflight summary ----
if RUN_PREFLIGHT:
    pf_backbone = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    failed = [r for r in PREFLIGHT_RESULTS if r[1] == "FAIL"]
    print(f"\nPreflight: {len(PREFLIGHT_RESULTS) - len(failed)} passed, {len(failed)} failed")
    for name, status, detail in PREFLIGHT_RESULTS:
        print(f"  {status}  {name}" + (f"  ({detail})" if status == "FAIL" else ""))
    if failed:
        raise RuntimeError("Preflight failed: " + "; ".join(f"{n}: {d}" for n, _, d in failed))

### 13.1 Tiny overfit test (optional)

Train on a handful of cases; loss should drop sharply. Catches broken labels/padding/masking before spending GPU time.

In [ ]:
if RUN_TINY_OVERFIT:
    tiny_overfit_test(exp_cfg)
else:
    print("RUN_TINY_OVERFIT=False: skipped.")

## 14. Main training experiment

Trains **only** `exp_cfg` (section 11) when `RUN_TRAINING = True`; saves `best_model.pt`, `metrics.json`,
`history.json` and `config.json` in `exp_cfg.run_dir`, and reloads the best checkpoint.

In [ ]:
model = None

if RUN_TRAINING:
    model, best_f1, history = fit(exp_cfg, train_loader, dev_loader)
else:
    print("RUN_TRAINING=False: training skipped.")

## 15. Knowledge distillation (optional)

`teacher (trained Phase-5 model) → per-statute logits → smaller student`

Each statute is an independent binary decision, so the soft target is per class:
`KD = T² · mean BCE(student/T, sigmoid(teacher/T))`; total = `(1 − α) · BCE(student, gold) + α · KD`.

Runs only when `RUN_KD = True`. Teacher: decoder + full fine-tuning loaded from `KD_TEACHER_CHECKPOINT`.
Student: 4-layer decoder + LORMA, initialised from the first `STUDENT_LAYERS` blocks of `KD_STUDENT_INIT_SOURCE`
(or resumed from `KD_STUDENT_RESUME_CHECKPOINT`).

In [ ]:
class DistillationExperiment:
    def __init__(self, teacher, student, kd_loss_fn, temperature=2.0, alpha=0.5, teacher_drop_keys=()):
        self.teacher = teacher
        self.student = student
        self.kd_loss_fn = kd_loss_fn
        self.temperature = temperature
        self.alpha = alpha
        self.teacher_drop_keys = set(teacher_drop_keys)

        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

    def compute_loss(self, tensors):
        with torch.no_grad():
            teacher_out = self.teacher(**{
                k: v for k, v in tensors.items()
                if k not in self.teacher_drop_keys
            })

        student_out = self.student(**tensors)
        gold_loss = student_out["loss"]
        kd_loss = self.kd_loss_fn(student_out, teacher_out, tensors, self.temperature)

        loss = (1.0 - self.alpha) * gold_loss + self.alpha * kd_loss
        return {"loss": loss, "gold_loss": gold_loss, "kd_loss": kd_loss}


def train_kd_epoch(
    kd_exp, loader, optimizer, device,
    grad_accum=1, max_grad_norm=1.0,
    use_amp=True, scaler=None, scheduler=None,
):
    kd_exp.student.train()
    kd_exp.teacher.eval()

    amp = _amp_enabled(use_amp)
    if scaler is None:
        scaler = make_scaler(amp)

    optimizer.zero_grad(set_to_none=True)
    tot = tot_gold = tot_kd = 0.0
    num_batches = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="kd", leave=False)):
        tensors, _ = split_batch(batch, device)

        remaining = num_batches - (step // grad_accum) * grad_accum
        current_accum = min(grad_accum, remaining)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            out = kd_exp.compute_loss(tensors)
            loss = out["loss"] / current_accum

        if not torch.isfinite(out["loss"]).all():
            print(
                f"Batch {step}:",
                "gold =", out["gold_loss"].item(),
                "kd =", out["kd_loss"].item(),
                "total =", out["loss"].item(),
                "lr =", optimizer.param_groups[0]["lr"],
            )
            raise RuntimeError(f"Non-finite KD loss at batch {step}")

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == num_batches:
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in kd_exp.student.parameters() if p.requires_grad],
                    max_grad_norm,
                )

            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()

            step_succeeded = not amp or scaler.get_scale() >= old_scale
            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None and step_succeeded:
                scheduler.step()

        tot += out["loss"].item()
        tot_gold += out["gold_loss"].item()
        tot_kd += out["kd_loss"].item()

    n = max(num_batches, 1)
    return tot / n, tot_gold / n, tot_kd / n

def build_truncated_decoder_backbone(n_layers, source=None):
    student_config = replace(decoder_config, n_layer=n_layers)
    backbone = DecoderOnlyTransformer(student_config)

    ckpt = _load_state(source or DECODER_CHECKPOINT)

    # Accept both Phase 1 backbone and Phase 5 classification checkpoints
    if any(k.startswith("backbone.") for k in ckpt):
        ckpt = {
            k.removeprefix("backbone."): v
            for k, v in ckpt.items()
            if k.startswith("backbone.")
        }

    # Keep only parameters belonging to the truncated student
    target_keys = backbone.state_dict()
    filtered = {
        k: v for k, v in ckpt.items()
        if k in target_keys and v.shape == target_keys[k].shape
    }

    # Every student parameter must actually be initialized
    missing = set(target_keys) - set(filtered)
    if missing:
        raise RuntimeError(
            f"Student initialization missing {len(missing)} tensors: "
            f"{sorted(missing)[:10]}"
        )

    backbone.load_state_dict(filtered, strict=True)

    print(
        f"Student decoder: {n_layers} layers | "
        f"Loaded {len(filtered)} tensors successfully"
    )

    return backbone


def multilabel_kd_loss(student_out, teacher_out, tensors, T):
    s = student_out["logits"].float()
    t = teacher_out["logits"].detach().float()

    if not torch.isfinite(s).all():
        raise RuntimeError("Student logits contain NaN/Inf")
    if not torch.isfinite(t).all():
        raise RuntimeError("Teacher logits contain NaN/Inf")

    with torch.autocast(device_type="cuda", enabled=False):
        soft_targets = torch.sigmoid(t / T)
        loss = F.binary_cross_entropy_with_logits(
            s / T, soft_targets
        ) * (T * T)

    return loss

In [ ]:
# ============================================================
# KD setup: FULL decoder teacher -> 4-layer LORMA decoder
# ============================================================

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5
KD_LR = 2e-4
KD_EPOCHS = 8
STUDENT_LAYERS = 4

# Teacher: decoder + full fine-tuning (loaded, not trained here).
teacher_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="full",
    learning_rate=5e-4,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=8,
    pooling_strategy="last_token",
    prediction_threshold=0.5,
    run_name="phase5_decoder_full",
)

# Student: 4-layer decoder + LORMA trained with the KD loss (fp32: use_amp=False, as in the original run).
student_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="lorma",
    learning_rate=KD_LR,
    batch_size=BATCH_SIZE,
    grad_accum=8,
    epochs=KD_EPOCHS,
    lorma_rank=16,
    lorma_alpha=32.0,
    pooling_strategy="last_token",
    prediction_threshold=0.5,
    use_amp=False,
    run_name="phase5_decoder_kd",
)

assert student_cfg.backbone_name == "decoder"




def _drop_loss_buffers(state):
    # pos_weight is a loss buffer, not a learned parameter
    state.pop("pos_weight", None)
    state.pop("loss_fn.pos_weight", None)
    return state


def build_kd_teacher(checkpoint_path=None):
    path = Path(checkpoint_path or KD_TEACHER_CHECKPOINT)
    if not path.is_file():
        raise FileNotFoundError(
            f"KD teacher checkpoint not found: {path}. Set KD_TEACHER_CHECKPOINT in section 2."
        )
    # Architecture only: all weights come from the teacher checkpoint.
    teacher = build_adapted_model(teacher_cfg, backbone=DecoderOnlyTransformer(decoder_config))
    teacher.load_state_dict(_drop_loss_buffers(torch.load(path, map_location="cpu", weights_only=True)), strict=True)
    teacher.eval()
    print("Teacher loaded:", path)
    return teacher


def build_kd_student(resume_checkpoint=None):
    resume = resume_checkpoint or KD_STUDENT_RESUME_CHECKPOINT
    if resume:
        backbone = DecoderOnlyTransformer(replace(decoder_config, n_layer=STUDENT_LAYERS))
    else:
        backbone = build_truncated_decoder_backbone(STUDENT_LAYERS, source=KD_STUDENT_INIT_SOURCE)
    student = build_adapted_model(student_cfg, backbone=backbone, pos_weight=pos_weight_for(student_cfg))
    if resume:
        state = _drop_loss_buffers(torch.load(resume, map_location="cpu", weights_only=True))
        student.load_state_dict(state, strict=True)
        print("Loaded KD student checkpoint:", resume)
    return student


def build_kd_pair():
    teacher = build_kd_teacher()     # fails early if the teacher checkpoint is missing
    student = build_kd_student()
    return teacher, student


def run_kd():
    teacher, student = build_kd_pair()

    kd_exp = DistillationExperiment(
        teacher,
        student,
        multilabel_kd_loss,
        temperature=KD_TEMPERATURE,
        alpha=KD_ALPHA,
        teacher_drop_keys=("labels", "has_labels"),
    )

    optimizer = build_optimizer(student, student_cfg)
    scheduler = build_scheduler(
        optimizer,
        student_cfg,
        len(train_loader),
    )

    scaler = make_scaler(_amp_enabled(student_cfg.use_amp))

    run_dir = student_cfg.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    save_experiment_config(student_cfg)

    best_f1 = -1.0

    for epoch in range(student_cfg.epochs):
        loss, gold, kd = train_kd_epoch(
            kd_exp,
            train_loader,
            optimizer,
            DEVICE,
            grad_accum=student_cfg.grad_accum,
            max_grad_norm=student_cfg.max_grad_norm,
            use_amp=student_cfg.use_amp,
            scaler=scaler,
            scheduler=scheduler,
        )

        val_out = run_validation(
            student,
            dev_loader,
            DEVICE,
            use_amp=student_cfg.use_amp,
        )

        metrics = compute_multilabel_metrics(
            val_out["labels"],
            logits_to_multi_hot(
                val_out["logits"],
                student_cfg.prediction_threshold,
            ),
        )

        print(
            f"epoch {epoch+1}: "
            f"loss={loss:.4f} "
            f"gold={gold:.4f} "
            f"kd={kd:.4f} "
            f"Macro-F1={metrics['macro_f1']:.4f}"
        )

        if metrics["macro_f1"] > best_f1:
            best_f1 = metrics["macro_f1"]

            torch.save(
                student.state_dict(),
                run_dir / "best_model.pt",
            )

            with open(run_dir / "metrics.json", "w") as f:
                json.dump(
                    {
                        **metrics,
                        "temperature": KD_TEMPERATURE,
                        "alpha": KD_ALPHA,
                        "student_layers": STUDENT_LAYERS,
                        "teacher": teacher_cfg.run_name,
                        "threshold": student_cfg.prediction_threshold,
                    },
                    f,
                    indent=2,
                )

    student.load_state_dict(
        torch.load(
            run_dir / "best_model.pt",
            map_location=DEVICE,
            weights_only=True,
        )
    )

    del kd_exp, teacher, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return student, best_f1


def load_kd_student(checkpoint_path=None):
    path = Path(checkpoint_path or (student_cfg.run_dir / "best_model.pt"))
    if not path.is_file():
        raise FileNotFoundError(f"KD student checkpoint not found: {path}")
    student = build_kd_student(resume_checkpoint=path)
    student.eval()
    return student


def kd_efficiency_report(student):
    """Parameter counts of teacher vs. student (teacher rebuilt from its checkpoint)."""
    teacher = build_kd_teacher()
    t_total = sum(p.numel() for p in teacher.parameters())
    s_total = sum(p.numel() for p in student.parameters())
    s_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
    del teacher
    gc.collect()
    report = {
        "teacher_total_params": t_total,
        "student_total_params": s_total,
        "student_trainable_params": s_train,
        "param_reduction_pct": round(100 * (1 - s_total / t_total), 2),
    }
    print(json.dumps(report, indent=2))
    return report

### 15.1 Run knowledge distillation

In [ ]:
kd_student = None

if RUN_KD:
    kd_student, kd_best_f1 = run_kd()
    print("Best KD Macro-F1:", kd_best_f1)
    kd_metadata = {
        "teacher": teacher_cfg.run_name,
        "teacher_checkpoint": str(KD_TEACHER_CHECKPOINT),
        "student_layers": STUDENT_LAYERS,
        "student_init": str(KD_STUDENT_RESUME_CHECKPOINT or KD_STUDENT_INIT_SOURCE),
        "temperature": KD_TEMPERATURE,
        "alpha": KD_ALPHA,
        "best_dev_macro_f1": kd_best_f1,
        **kd_efficiency_report(kd_student),
    }
    with open(student_cfg.run_dir / "kd_metadata.json", "w", encoding="utf-8") as f:
        json.dump(kd_metadata, f, indent=2)
else:
    print("RUN_KD=False: knowledge distillation skipped.")

## 16. Test inference (dev-tuned threshold)

`INFERENCE_SOURCE = "main"` uses the `exp_cfg` model (trained above, or loaded from `INFERENCE_CHECKPOINT` /
`exp_cfg.run_dir/best_model.pt`); `"kd"` uses the KD student. A single global threshold is tuned on dev
(`tune_threshold`, grid 0.05–0.95) and used for the test predictions. Each source writes to its own run directory.

In [ ]:
if RUN_INFERENCE:
    if INFERENCE_SOURCE == "main":
        inference_cfg = exp_cfg
        inference_model = model if model is not None else load_trained_model(exp_cfg, INFERENCE_CHECKPOINT)
    elif INFERENCE_SOURCE == "kd":
        inference_cfg = student_cfg
        inference_model = kd_student if kd_student is not None else load_kd_student(INFERENCE_CHECKPOINT)
    else:
        raise ValueError(f"INFERENCE_SOURCE must be 'main' or 'kd', got {INFERENCE_SOURCE!r}")

    inference_model = inference_model.to(DEVICE).eval()
    print("Inference source:", INFERENCE_SOURCE, "| run:", inference_cfg.run_dir.name)

    test_predictions, best_threshold, dev_metrics = predict_test(inference_model, inference_cfg)
    print("Mean statutes / case:", np.mean([len(p["pred"]) for p in test_predictions]),
          "| cases with no statute:", sum(1 for p in test_predictions if not p["pred"]))
else:
    print("RUN_INFERENCE=False: test inference skipped.")

## 17. Submission validation and ZIP

Expected `prediction.json` — a JSON **list**, one object per test case; `pred` = unique integer statute IDs:

```json
[
  {"id": "case_001", "pred": [1, 7, 23]},
  {"id": "case_002", "pred": [4]}
]
```

In [ ]:
if RUN_INFERENCE:
    validate_submission_format(
        test_predictions,
        official_test_ids=[case["id"] for case in test_cases],
        valid_statutes=set(STATUTE_IDS),
    )

    pred_path, zip_path = save_submission(test_predictions, inference_cfg.run_dir)
    verify_zip_root(zip_path)
    with open(inference_cfg.run_dir / "threshold.json", "w") as f:
        json.dump({"threshold": float(best_threshold), "dev_macro_f1": dev_metrics["macro_f1"]}, f, indent=2)
    print("Final submission ready:", zip_path)
else:
    print("RUN_INFERENCE=False: no submission written.")

## Recorded Codabench results (Phase 5)

| Run | Backbone | Adaptation | Token pooling | pos_weight | Codabench Macro-F1 | Mean statutes / test case |
|---|---|---|---|---|---|---|
| cnn lorma | cnn | LORMA (attention + FFN layers) | mean | no | **0.2846** (leaderboard submission) | 6.19 |
| dec lorma | decoder | LORMA | last_token | yes | 0.2708 | 6.20 |
| dec full | decoder | full fine-tuning | last_token | yes | 0.2670 | 5.74 |
| KD | decoder, 4-layer student | KD (teacher: decoder full FT) + LORMA | last_token | no | 0.2553 | 7.99 |

